#### Dependencies

In [61]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lifelines import KaplanMeierFitter, CoxPHFitter, NelsonAalenFitter
from sklearn.model_selection import KFold
from scipy.stats import gaussian_kde, norm
from sklearn.model_selection import train_test_split
from matplotlib.ticker import FuncFormatter
from lifelines.plotting import add_at_risk_counts

from __future__ import annotations
from collections import Counter
from dataclasses import dataclass
import hashlib

import warnings
warnings.filterwarnings("ignore")


from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score


from lifelines.statistics import logrank_test


from going_modular import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None) 


from rpy2.robjects import Formula
import rpy2.robjects as ro
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import default_converter
from rpy2.robjects import r
from rpy2.robjects import pandas2ri, conversion
from rpy2.robjects.packages import importr
from rpy2.robjects.vectors import DataFrame, FloatVector, IntVector, ListVector, Vector, StrVector

rstpm2 = importr("rstpm2")
survival = importr("survival")
ggplot2 = importr("ggplot2")
graphics = importr("graphics")
stats = importr("stats")
lmtest = importr("lmtest")


#### Datasets

In [62]:
lical0 = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFile/lical0_processed_data_for_fp_model_21-01-2026.csv')
miro0 = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFile/miro0_processed_data_for_fp_model_21-01-2026.csv')
ril_3010 = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFile/ril_3010_processed_data_for_fp_model_21-01-2026.csv')

# miroli0 = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFile/miroli0_processed_data_for_fp_model_21-01-2026.csv')
proact0 = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFile/proact0_processed_data_for_fp_model_21-01-2026.csv')

MND_lica = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFIle/MNDRegisterDataset_licals.csv')
MND_miro = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFIle/MNDRegisterDataset_mirocals.csv')
MND_rilu = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFIle/MNDRegisterDataset_riluzole.csv')

In [63]:
MND_lica.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   subject_id          146 non-null    int64  
 1   Event               146 non-null    int64  
 2   Disease_Duration    146 non-null    float64
 3   Age                 146 non-null    float64
 4   Diagnostic_Delay    146 non-null    float64
 5   Sex_Male            146 non-null    int64  
 6   Onset_Limb          146 non-null    int64  
 7   Sex_onset           146 non-null    int64  
 8   Age_Sex             146 non-null    float64
 9   Age_onset           146 non-null    float64
 10  Age_sq              146 non-null    float64
 11  TRICALS_Risk_Score  146 non-null    float64
dtypes: float64(7), int64(5)
memory usage: 13.8 KB


In [64]:
# MND_rilu.info()

In [65]:
proact0 = proact0[proact0['Disease_Duration'] < 120]

##### **Proact with eligibility criteria**

In [66]:
proact_miro = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFIle/proact_miro_full.csv')
proact_lica = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFIle/proact_lica_full.csv')

proact_miro['slope'] = (48 - proact_miro['ALSFRS_RT']) / proact_miro['Disease_Duration']
proact_lica['slope'] = (48 - proact_lica['ALSFRS_RT']) / proact_lica['Disease_Duration']

binss = [-np.inf, 0.31, 1.17, np.inf]
labelss = ["Slow", "Intermediate", "Fast"]

proact_miro["Progression_group"] = pd.cut(proact_miro["slope"], bins=binss, labels=labelss)
proact_lica["Progression_group"] = pd.cut(proact_lica["slope"], bins=binss, labels=labelss)


proact_miro['Study_Arm_Placebo'] = (proact_miro['Study_Arm'] == 'Placebo').astype(int)
proact_lica['Study_Arm_Placebo'] = (proact_lica['Study_Arm'] == 'Placebo').astype(int)

proact_miro = proact_miro[['subject_id', 'Event', 'Sex_Male', 'Onset_Limb', 'Study_Arm_Placebo', 'Age_std', 'Diagnosis_delay_l_std', 'Disease_Duration', 'TRICALS', 'Sex_onset',	'Age_Sex', 'Age_onset', 'Age_sq', "Progression_group"]]
proact_lica = proact_lica[['subject_id', 'Event', 'Sex_Male', 'Onset_Limb', 'Study_Arm_Placebo', 'Age_std', 'Diagnosis_delay_l_std', 'Disease_Duration', 'TRICALS', 'Sex_onset',	'Age_Sex', 'Age_onset', 'Age_sq', "Progression_group"]]

proact_miro = proact_miro.rename(columns={'Age_std': 'Age', 'Diagnosis_delay_l_std': 'Diagnostic_Delay'})
proact_lica = proact_lica.rename(columns={'Age_std': 'Age', 'Diagnosis_delay_l_std': 'Diagnostic_Delay'})


# print(f"proact_miro: {proact_miro.shape} | proact_lica: {proact_lica.shape}")
proact_lica.head(2)

,subject_id,Event,Sex_Male,Onset_Limb,Study_Arm_Placebo,Age,Diagnostic_Delay,Disease_Duration,TRICALS,Sex_onset,Age_Sex,Age_onset,Age_sq,Progression_group
0,1333,1,0,1,0,-0.428923,0.605487,22.996058,-3.779545,0,-0.000000,-0.428923,0.183975,Fast
1,3350,0,1,1,0,1.305237,0.462062,21.747700,-4.907534,1,1.305237,1.305237,1.703642,Intermediate


In [67]:
# proact_lica.info()

#### **Select only the treatment arm**

In [68]:
lical0_trt = lical0[lical0['Study_Arm_Placebo'] == 0].copy()
miro0_trt = miro0[miro0['Study_Arm_Placebo'] == 0].copy()
# miroli0_trt = miroli0[miroli0['Study_Arm_Placebo'] == 0].copy()
ril_3010_trt = ril_3010[ril_3010['Study_Arm_Placebo'] == 0].copy()
proact0_trt = proact0[proact0['Study_Arm_Placebo'] == 0].copy() 

In [69]:
# proact0_trt.info()

#### **Select only the placebo arm**

In [70]:
lical0_pla = lical0[lical0['Study_Arm_Placebo'] == 1].copy()
miro0_pla = miro0[miro0['Study_Arm_Placebo'] == 1].copy()
# miroli0_pla = miroli0[miroli0['Study_Arm_Placebo'] == 1].copy()
ril_3010_pla = ril_3010[ril_3010['Study_Arm_Placebo'] == 1].copy()
proact0_pla = proact0[proact0['Study_Arm_Placebo'] == 1].copy() 

In [71]:
proact_miro_pla = proact_miro[proact_miro['Study_Arm_Placebo'] == 1]
proact_lica_pla = proact_lica[proact_lica['Study_Arm_Placebo'] == 1]

proact_miro_trt = proact_miro[proact_miro['Study_Arm_Placebo'] == 0]
proact_lica_trt = proact_lica[proact_lica['Study_Arm_Placebo'] == 0]

print(f"proact_miro_trt: {proact_miro_trt.shape} | proact_lica_trt: {proact_lica_trt.shape}\n")
print(f"proact_miro_pla: {proact_miro_pla.shape} | proact_lica_pla: {proact_lica_pla.shape}\n")
proact_lica_pla.head(2)

proact_miro_trt: (240, 14) | proact_lica_trt: (346, 14)

proact_miro_pla: (188, 14) | proact_lica_pla: (281, 14)



,subject_id,Event,Sex_Male,Onset_Limb,Study_Arm_Placebo,Age,Diagnostic_Delay,Disease_Duration,TRICALS,Sex_onset,Age_Sex,Age_onset,Age_sq,Progression_group
3,4753,1,0,1,1,-1.902391,-1.024997,14.586071,-2.589279,0,-0.0,-1.902391,3.619093,Fast
5,7540,0,0,1,1,0.704515,0.685898,20.400788,-4.593248,0,0.0,0.704515,0.496341,Intermediate


In [72]:
print(f"Miro: {miro0_pla.Disease_Duration.describe().round(2).to_dict()}\n")
# print(f"miroli: {miroli0_pla.Disease_Duration.describe().round(2).to_dict()}\n")
print(f"Lical :{lical0_pla.Disease_Duration.describe().round(2).to_dict()}\n")
print(f"Riluzole: {ril_3010_pla.Disease_Duration.describe().round(2).to_dict()}\n")

print(f"Proact: {proact0_pla.Disease_Duration.describe().round(2).to_dict()}\n")

print(f"MND_lica: {MND_lica.Disease_Duration.describe().round(2).to_dict()}\n")
print(f"MND_miro: {MND_miro.Disease_Duration.describe().round(2).to_dict()}\n")
print(f"MND_rilu: {MND_rilu.Disease_Duration.describe().round(2).to_dict()}\n")

Miro: {'count': 110.0, 'mean': 29.04, 'std': 8.09, 'min': 8.05, '25%': 25.14, '50%': 29.35, '75%': 34.32, 'max': 44.92}

Lical :{'count': 87.0, 'mean': 31.76, 'std': 8.63, 'min': 10.86, '25%': 24.95, '50%': 34.81, '75%': 37.67, 'max': 46.14}

Riluzole: {'count': 242.0, 'mean': 13.41, 'std': 5.41, 'min': 0.3, '25%': 9.2, '50%': 15.82, '75%': 17.94, 'max': 20.7}

Proact: {'count': 1547.0, 'mean': 29.64, 'std': 11.36, 'min': 4.66, '25%': 21.83, '50%': 27.9, '75%': 35.41, 'max': 100.1}

MND_lica: {'count': 146.0, 'mean': 23.87, 'std': 7.56, 'min': 7.13, '25%': 18.32, '50%': 24.56, '75%': 30.42, 'max': 35.94}

MND_miro: {'count': 92.0, 'mean': 18.8, 'std': 5.62, 'min': 5.22, '25%': 14.82, '50%': 19.73, '75%': 24.08, 'max': 27.92}

MND_rilu: {'count': 169.0, 'mean': 29.55, 'std': 11.92, 'min': 5.65, '25%': 20.86, '50%': 29.53, '75%': 36.24, 'max': 59.92}



#### **Choosing cohorts**

In [73]:
MND_miro['Study_Arm_Placebo'] = 1
MND_lica['Study_Arm_Placebo'] = 1

MND_miro = MND_miro.rename(columns={'TRICALS_Risk_Score': 'TRICALS'})
MND_lica = MND_lica.rename(columns={'TRICALS_Risk_Score': 'TRICALS'})

miro0_trt = miro0_trt[['subject_id', 'Event', 'Disease_Duration','Study_Arm_Placebo', 'Age', 'Diagnostic_Delay',
       'Sex_Male', 'Onset_Limb', 'Sex_onset', 'Age_Sex', 'Age_onset', 'Age_sq', 'TRICALS']]
lical0_trt = lical0_trt[['subject_id', 'Event', 'Disease_Duration','Study_Arm_Placebo', 'Age', 'Diagnostic_Delay',
       'Sex_Male', 'Onset_Limb', 'Sex_onset', 'Age_Sex', 'Age_onset', 'Age_sq', 'TRICALS']]


miro_trt_trical = miro0_trt[(miro0_trt['TRICALS'] >= -6) & (miro0_trt['TRICALS'] <= -2)]
MND_miro_trical = MND_miro[(MND_miro['TRICALS'] >= -6) & (MND_miro['TRICALS'] <= -2)]

lica_trt_trical = lical0_trt[(lical0_trt['TRICALS'] >= -6) & (lical0_trt['TRICALS'] <= -2)]
MND_lica_trical = MND_lica[(MND_lica['TRICALS'] >= -6) & (MND_lica['TRICALS'] <= -2)]


In [74]:
proact_lica_pla = proact_lica_pla.drop(columns=['Progression_group'])
proact_miro_pla = proact_miro_pla.drop(columns=['Progression_group'])

In [75]:
# combine mirocals treated and licals placebo
comb = pd.concat([lical0_trt, proact_lica_pla], axis=0, ignore_index=True)
comb['Group'] = (comb['Study_Arm_Placebo'] == 0).astype(int)
comb.shape

(370, 14)

In [76]:
comb.head(2)

,subject_id,Event,Disease_Duration,Study_Arm_Placebo,Age,Diagnostic_Delay,Sex_Male,Onset_Limb,Sex_onset,Age_Sex,Age_onset,Age_sq,TRICALS,Group
0,P01003,0,34.571616,0,-1.105049,0.017917,1,1,1,-1.105049,-1.105049,1.221133,-1.025245,1
1,P01004,0,28.428384,0,-1.059473,-1.452295,1,1,1,-1.059473,-1.059473,1.122482,-0.706113,1


In [77]:
# comb.info()

In [78]:
# ============================================================
# PROPENSITY SCORE + OVERLAP
# ============================================================
def fit_propensity_score(
    df: pd.DataFrame,
    covariates: List[str],
    penalty: str = "l2",
    C: float = 1.0
) -> Tuple[pd.DataFrame, Pipeline]:
    """
    Logistic PS model: probability of being in treated trial arm.
    Missing numeric values are median-imputed.
    """
    x = df[covariates].copy()
    y = df["Group"].astype(int).values

    numeric_features = covariates
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features)
        ],
        remainder="drop"
    )

    model = LogisticRegression(
        penalty=penalty,
        C=C,
        solver="lbfgs",
        max_iter=5000
    )

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(x, y)
    ps = pipe.predict_proba(x)[:, 1]

    out = df.copy()
    out["ps"] = ps
    out["ps_logit"] = np.log(np.clip(ps, 1e-6, 1-1e-6) / np.clip(1-ps, 1e-6, 1-1e-6))

    auc = roc_auc_score(y, ps)
    print(f"Propensity model AUC: {auc:.3f}")

    return out, pipe

In [79]:
covariates = ['Disease_Duration', 'Age', 'Diagnostic_Delay', 'Onset_Limb', 'Sex_Male'] #, 'Vital_capacity']

comb_ps, propen = fit_propensity_score(comb, covariates)
# 
comb_ps.head(2)

Propensity model AUC: 0.759


,subject_id,Event,Disease_Duration,Study_Arm_Placebo,Age,Diagnostic_Delay,Sex_Male,Onset_Limb,Sex_onset,Age_Sex,Age_onset,Age_sq,TRICALS,Group,ps,ps_logit
0,P01003,0,34.571616,0,-1.105049,0.017917,1,1,1,-1.105049,-1.105049,1.221133,-1.025245,1,0.533542,0.134372
1,P01004,0,28.428384,0,-1.059473,-1.452295,1,1,1,-1.059473,-1.059473,1.122482,-0.706113,1,0.505309,0.021238


In [80]:
comb_ps.shape

(370, 16)

In [81]:
# Trim non overlapping records
def trim_nonoverlap(df: pd.DataFrame, ps_col: str = "ps") -> pd.DataFrame:
    """
    Trim non-overlapping tails:
      treated above max(control) and control below min(treated)
    """
    x = df.copy()
    ps_t = x.loc[x["Group"] == 1, ps_col]
    ps_c = x.loc[x["Group"] == 0, ps_col]

    lower = ps_t.min()
    upper = ps_c.max()

    # More symmetric overlap region
    lower_common = max(ps_t.min(), ps_c.min())
    upper_common = min(ps_t.max(), ps_c.max())

    trimmed = x[(x[ps_col] >= lower_common) & (x[ps_col] <= upper_common)].copy()

    print(f"Before trimming: {len(x)}")
    print(f"After trimming:  {len(trimmed)}")
    print(f"Removed:         {len(x) - len(trimmed)}")

    return trimmed.reset_index(drop=True)

trim_propen = trim_nonoverlap(comb_ps, "ps")
# trim_propen

Before trimming: 370
After trimming:  334
Removed:         36


In [82]:
trim_propen.shape
# trim_propen.head()

(334, 16)

In [83]:
# sns.kdeplot(data=comb_ps, x="ps", hue="Group", fill=True)

# plt.title("Density Plot of Two Groups")
# plt.xlabel("Value")
# plt.ylabel("Density")
# plt.show()

In [84]:
# sns.kdeplot(data=trim_propen, x="ps", hue="Group", fill=True)

# plt.title("Density Plot of Two Groups")
# plt.xlabel("Value")
# plt.ylabel("Density")
# plt.show()

In [85]:
# sns.histplot(data=comb_ps,x="ps",hue="Group",bins=20,stat="probability")

# plt.title("Density Plot of Two Groups")
# plt.xlabel("Value")
# plt.ylabel("Density")
# plt.show()

In [86]:
# sns.histplot(data=trim_propen,x="ps",hue="Group",bins=20,stat="probability")

# plt.title("Density Plot of Two Groups")
# plt.xlabel("Value")
# plt.ylabel("Density")
# plt.show()

In [87]:
# make treated and contral groups more comparable
def add_weights(df: pd.DataFrame, method: str = "overlap", ps_col: str = "ps") -> pd.DataFrame:
    """
    Supported:
      - overlap
      - iptw
      - unstabilized
    """
    x = df.copy()
    ps = np.clip(x[ps_col].values, 1e-6, 1 - 1e-6)
    g = x["Group"].values

    if method == "overlap":
        w = np.where(g == 1, 1 - ps, ps)
    elif method in ("iptw", "unstabilized"):
        w = np.where(g == 1, 1 / ps, 1 / (1 - ps))
    else:
        raise ValueError("method must be 'overlap' or 'iptw'")

    x["weight"] = w
    return x

comb_weight = add_weights(trim_propen)
comb_weight.head(2)

,subject_id,Event,Disease_Duration,Study_Arm_Placebo,Age,Diagnostic_Delay,Sex_Male,Onset_Limb,Sex_onset,Age_Sex,Age_onset,Age_sq,TRICALS,Group,ps,ps_logit,weight
0,P01003,0,34.571616,0,-1.105049,0.017917,1,1,1,-1.105049,-1.105049,1.221133,-1.025245,1,0.533542,0.134372,0.466458
1,P01004,0,28.428384,0,-1.059473,-1.452295,1,1,1,-1.059473,-1.059473,1.122482,-0.706113,1,0.505309,0.021238,0.494691


In [88]:
# comb_weight.info()

In [89]:
# ============================================================
# BALANCE DIAGNOSTICS
# ============================================================
def weighted_mean(x, w):
    return np.sum(w * x) / np.sum(w)


def weighted_var(x, w):
    mu = weighted_mean(x, w)
    return np.sum(w * (x - mu) ** 2) / np.sum(w)


def smd_continuous(x_t, x_c, w_t=None, w_c=None):
    x_t = np.asarray(x_t, dtype=float)
    x_c = np.asarray(x_c, dtype=float)

    if w_t is None:
        mt = np.nanmean(x_t)
        vt = np.nanvar(x_t, ddof=0)
    else:
        mt = weighted_mean(x_t, w_t)
        vt = weighted_var(x_t, w_t)

    if w_c is None:
        mc = np.nanmean(x_c)
        vc = np.nanvar(x_c, ddof=0)
    else:
        mc = weighted_mean(x_c, w_c)
        vc = weighted_var(x_c, w_c)

    pooled_sd = np.sqrt((vt + vc) / 2)
    if pooled_sd == 0 or np.isnan(pooled_sd):
        return np.nan
    return (mt - mc) / pooled_sd


def smd_binary(x_t, x_c, w_t=None, w_c=None):
    x_t = np.asarray(x_t, dtype=float)
    x_c = np.asarray(x_c, dtype=float)

    if w_t is None:
        pt = np.nanmean(x_t)
    else:
        pt = weighted_mean(x_t, w_t)

    if w_c is None:
        pc = np.nanmean(x_c)
    else:
        pc = weighted_mean(x_c, w_c)

    p = (pt + pc) / 2
    denom = np.sqrt(p * (1 - p))
    if denom == 0 or np.isnan(denom):
        return np.nan
    return (pt - pc) / denom


def balance_table(df: pd.DataFrame, covariates: List[str], weight_col: Optional[str] = None) -> pd.DataFrame:
    rows = []

    treated = df[df["Group"] == 1].copy()
    control = df[df["Group"] == 0].copy()

    for col in covariates:
        xt = treated[col].astype(float)
        xc = control[col].astype(float)

        mask_t = xt.notna()
        mask_c = xc.notna()
        xt = xt[mask_t]
        xc = xc[mask_c]

        wt = treated.loc[mask_t, weight_col].values if weight_col is not None else None
        wc = control.loc[mask_c, weight_col].values if weight_col is not None else None

        uniq = pd.concat([xt, xc]).dropna().unique()
        is_binary = set(np.unique(uniq)).issubset({0.0, 1.0})

        if is_binary:
            smd = smd_binary(xt, xc, wt, wc)
        else:
            smd = smd_continuous(xt, xc, wt, wc)

        if weight_col is None:
            mt = np.nanmean(xt)
            mc = np.nanmean(xc)
        else:
            mt = weighted_mean(xt.values, wt)
            mc = weighted_mean(xc.values, wc)

        rows.append({
            "Variable": col,
            "Treated_mean": mt,
            "Control_mean": mc,
            "SMD": smd,
            "Abs_SMD": np.abs(smd)
        })

    out = pd.DataFrame(rows).sort_values("Abs_SMD", ascending=False).reset_index(drop=True)
    return out

comb_baln = balance_table(comb_weight, covariates, 'weight')
comb_baln

,Variable,Treated_mean,Control_mean,SMD,Abs_SMD
0,Diagnostic_Delay,-0.281540,-0.227866,-0.048286,0.048286
1,Disease_Duration,27.047856,27.302797,-0.032396,0.032396
2,Age,0.024282,0.046175,-0.021067,0.021067
3,Onset_Limb,0.805905,0.798151,0.019459,0.019459
4,Sex_Male,0.719860,0.714229,0.012503,0.012503


In [90]:
# comb_weight.isna().sum()

In [91]:
# ============================================================
# SURVIVAL ANALYSIS
# ============================================================
def weighted_km_plot(
    df: pd.DataFrame,
    time_col: str = "Time",
    event_col: str = "Event",
    weight_col: Optional[str] = "weight",
    title: str = "Weighted Kaplan-Meier",
    x_max: Optional[float] = None
):
    plt.figure(figsize=(5, 4))

    km_t = KaplanMeierFitter()
    km_c = KaplanMeierFitter()

    treated = df[df["Group"] == 1].copy()
    control = df[df["Group"] == 0].copy()

    km_t.fit(
        durations=treated[time_col],
        event_observed=treated[event_col],
        weights=treated[weight_col] if weight_col else None,
        label="Treated"
    )
    km_c.fit(
        durations=control[time_col],
        event_observed=control[event_col],
        weights=control[weight_col] if weight_col else None,
        label="External control"
    )

    ax = km_t.plot_survival_function(ci_show=True)
    km_c.plot_survival_function(ci_show=True, ax=ax)

    ax.set_xlabel("Disease Duration")
    ax.set_ylabel("Survival probability")
    ax.set_title(title)
    ax.set_ylim(0,1.02)
    ax.grid(True, alpha=0.3)
    if x_max is not None:
        ax.set_xlim(0, x_max)
    plt.tight_layout()
    plt.show()
    

In [94]:
# weighted_km_plot(comb_weight, time_col = "Disease_Duration", event_col = "Event",weight_col = None,title = "Unweighted Kaplan-Meier")

In [95]:
# weighted_km_plot(comb_weight, time_col = "Disease_Duration", event_col = "Event",weight_col = "weight")

#### **Bootstrapping, calibration slope, regression coefficients, CI.**

##### mistakes

In [ ]:
# # ============================================================
# # HELPERS (DROP-IN): automatic direction detection + consistent
# # scoring for C-index, slope, shrinkage, and reporting.
# #
# # Key conventions enforced:
# #   - LP is extracted from stpm2 via type="lpmatrix" %*% coef
# #   - A "risk score" is constructed so that:
# #         mean(score | Event=1) > mean(score | Event=0)
# #     i.e. higher score => higher hazard (worse prognosis)
# #   - Concordance (Harrell's C) is computed using I(-score),
# #     because survival::concordance assumes larger x => longer survival.
# #   - Calibration slope is computed with score (higher => higher hazard).
# #   - CITL at t0 is computed from predicted survival and KM; independent of score sign.
# #
# # These helpers are designed to eliminate all of the sign flip confusion
# # you encountered (TRICALS changing the sign, etc.).
# # ============================================================
# # ============================================================
# # R snippets setup

# R = ro.r

# # ---------- R snippets ----------
# _r_conc_extract = R("""function(conc_obj) as.numeric(conc_obj$concordance[1])""")

# _r_predict_surv_t0 = R("""
# function(model, newdata, t0){
#   n <- nrow(newdata)
#   tmp <- transform(newdata, t0=rep(as.numeric(t0), n))
#   as.numeric(predict(model, newdata=tmp, type="surv", timevar="t0"))
# }
# """)

# _r_km_surv_t0_w = R("""
# function(df, t0, weight_col="weight"){
#   w <- as.numeric(df[[weight_col]])
#   fit <- survival::survfit(
#     survival::Surv(Disease_Duration, Event==1) ~ 1,
#     data=df,
#     weights=w
#   )
#   ss <- summary(fit, times=as.numeric(t0))
#   if(length(ss$surv)==0) return(NA_real_)
#   as.numeric(ss$surv[1])
# }
# """)


# _r_km_risk_ci_t0 = R("""
# function(df, t0){
#   fit <- survival::survfit(survival::Surv(Disease_Duration, Event==1) ~ 1, data=df)
#   ss <- summary(fit, times=as.numeric(t0))
#   if(length(ss$surv)==0) return(c(NA_real_, NA_real_, NA_real_))
#   S  <- as.numeric(ss$surv[1])
#   Lo <- as.numeric(ss$lower[1])
#   Hi <- as.numeric(ss$upper[1])
#   risk    <- 1 - S
#   risk_lo <- 1 - Hi
#   risk_hi <- 1 - Lo
#   c(risk, risk_lo, risk_hi)
# }
# """)


# # --- R function: IPCW recalibration at t0 ---
# _r_ipcw_recal_t0 = R("""
# function(time, status, p_pred, t0, estimate_slope=FALSE){
#   # time: follow-up time
#   # status: 1=event, 0=censored
#   # p_pred: predicted risk at t0 (individual)
#   # t0: horizon
#   # estimate_slope: if TRUE, fit intercept + slope; else intercept only with offset

#   time   <- as.numeric(time)
#   status <- as.integer(status)
#   status <- ifelse(status == 1L, 1L, 0L)
#   status <- as.integer(status)

#   p_pred <- as.numeric(p_pred)
#   t0     <- as.numeric(t0)

#   # clip predicted risks to avoid +/-Inf logits
#   eps <- 1e-12
#   p_pred <- pmin(pmax(p_pred, eps), 1 - eps)

#   # --- estimate censoring survival G(t) via KM on censoring indicator ---
#   # censoring event: 1 if censored, 0 otherwise
#   censor_event <- 1L - status
#   fitG <- survival::survfit(survival::Surv(time, censor_event) ~ 1)

#   # helper to get G(u) at specific times u using summary
#   G_at <- function(u){
#     ss <- summary(fitG, times=u)
#     if(length(ss$surv)==0){
#       # if u is beyond last event time, KM stays at last survival
#       return(tail(fitG$surv, 1))
#     }
#     return(ss$surv[1])
#   }

#   # build weights
#   w <- rep(0, length(time))
#   Y <- ifelse(time <= t0 & status == 1L, 1L, 0L)
#   Y <- as.integer(Y)
#   w <- as.numeric(w)

#   # Y <- as.integer(time <= t0 & status == 1L)

#   # events before/at t0: w = 1 / G(Ti)
#   idx_event <- which(time <= t0 & status == 1L)
#   if(length(idx_event) > 0){
#     Gi <- sapply(time[idx_event], G_at)
#     w[idx_event] <- 1 / pmax(Gi, eps)
#   }

#   # event-free at t0 (known): time >= t0 => w = 1 / G(t0)
#   idx_free <- which(time >= t0)
#   if(length(idx_free) > 0){
#     Gt0 <- G_at(t0)
#     w[idx_free] <- 1 / pmax(Gt0, eps)
#   }

#   # fit logistic recalibration
#   lp_pred <- qlogis(p_pred)

#   if(!estimate_slope){
#     fit <- stats::glm(Y ~ 1 + offset(lp_pred),
#                       family=stats::quasibinomial(),
#                       weights=w)
#     alpha <- stats::coef(fit)[1]
#     return(list(alpha=as.numeric(alpha),
#                 model=fit))
#   } else {
#     fit <- stats::glm(Y ~ lp_pred,
#                       family=stats::quasibinomial(),
#                       weights=w)
#     alpha <- stats::coef(fit)[1]
#     beta  <- stats::coef(fit)[2]
#     return(list(alpha=as.numeric(alpha),
#                 beta=as.numeric(beta),
#                 model=fit))
#   }
# }
# """)


# _r_ipcw_recal_t0_w = R("""
# function(time, status, p_pred, t0, base_w=NULL, estimate_slope=FALSE){
#   time   <- as.numeric(time)
#   status <- as.integer(status)
#   status <- ifelse(status == 1L, 1L, 0L)
#   status <- as.integer(status)

#   p_pred <- as.numeric(p_pred)
#   t0     <- as.numeric(t0)

#   if(is.null(base_w)){
#     base_w <- rep(1, length(time))
#   } else {
#     base_w <- as.numeric(base_w)
#   }

#   eps <- 1e-12
#   p_pred <- pmin(pmax(p_pred, eps), 1 - eps)

#   censor_event <- 1L - status
#   fitG <- survival::survfit(survival::Surv(time, censor_event) ~ 1)

#   G_at <- function(u){
#     ss <- summary(fitG, times=u)
#     if(length(ss$surv)==0){
#       return(tail(fitG$surv, 1))
#     }
#     return(ss$surv[1])
#   }

#   Y <- ifelse(time <= t0 & status == 1L, 1L, 0L)
#   Y <- as.integer(Y)

#   w_ipcw <- rep(0, length(time))

#   idx_event <- which(time <= t0 & status == 1L)
#   if(length(idx_event) > 0){
#     Gi <- sapply(time[idx_event], G_at)
#     w_ipcw[idx_event] <- 1 / pmax(Gi, eps)
#   }

#   idx_free <- which(time >= t0)
#   if(length(idx_free) > 0){
#     Gt0 <- G_at(t0)
#     w_ipcw[idx_free] <- 1 / pmax(Gt0, eps)
#   }

#   w <- base_w * w_ipcw
#   lp_pred <- qlogis(p_pred)

#   if(!estimate_slope){
#     fit <- stats::glm(Y ~ 1 + offset(lp_pred),
#                       family=stats::quasibinomial(),
#                       weights=w)
#     alpha <- stats::coef(fit)[1]
#     return(list(alpha=as.numeric(alpha), model=fit))
#   } else {
#     fit <- stats::glm(Y ~ lp_pred,
#                       family=stats::quasibinomial(),
#                       weights=w)
#     alpha <- stats::coef(fit)[1]
#     beta  <- stats::coef(fit)[2]
#     return(list(alpha=as.numeric(alpha), beta=as.numeric(beta), model=fit))
#   }
# }
# """)

# _r_km_risk_ci_t0_robust_w = R("""
# function(df, t0, weight_col="weight"){
#   w <- as.numeric(df[[weight_col]])
#   fit <- survival::survfit(
#     survival::Surv(Disease_Duration, Event==1) ~ 1,
#     data=df,
#     weights=w
#   )

#   tt <- fit$time
#   if(length(tt)==0) return(c(NA_real_, NA_real_, NA_real_))
#   t_use <- max(tt[tt <= as.numeric(t0)], na.rm=TRUE)
#   if(!is.finite(t_use)) return(c(NA_real_, NA_real_, NA_real_))

#   ss <- summary(fit, times=t_use)
#   if(length(ss$surv)==0) return(c(NA_real_, NA_real_, NA_real_))

#   S  <- as.numeric(ss$surv[1])
#   Lo <- as.numeric(ss$lower[1])
#   Hi <- as.numeric(ss$upper[1])

#   risk    <- 1 - S
#   risk_lo <- 1 - Hi
#   risk_hi <- 1 - Lo
#   c(risk, risk_lo, risk_hi)
# }
# """)

# def km_risk_ci_at_t0(r_df_sub, t0_months: int, weight_col: str = "weight") -> tuple[float, float, float]:
#     out = np.asarray(_r_km_risk_ci_t0_robust_w(r_df_sub, float(t0_months), weight_col), dtype=float)
#     if np.any(~np.isfinite(out)):
#         return (np.nan, np.nan, np.nan)
#     return (float(out[0]), float(out[1]), float(out[2]))


# def ipcw_calibration_intercept_t0_stpm2(
#     model,
#     r_df,
#     t0_months: float,
#     event_col: str = "Event",
#     time_col: str = "Disease_Duration",
# ) -> float:
#     # predicted risk at t0 from stpm2 (sign independent)
#     S_pred = predict_surv_at_t0(model, r_df, t0_months)
#     p_pred = 1.0 - np.asarray(S_pred, float)
#     p_pred = np.clip(p_pred, 1e-12, 1 - 1e-12)

#     # Extract from R df -> numpy, then wrap as R vectors
#     time_np   = np.asarray(r_df.rx2(time_col), dtype=float)
#     status_np = np.asarray(r_df.rx2(event_col), dtype=int)

#     time_r   = FloatVector(time_np.tolist())
#     status_r = IntVector(status_np.tolist())
#     p_pred_r = FloatVector(p_pred.tolist())

#     out = _r_ipcw_recal_t0(time_r, status_r, p_pred_r, float(t0_months), False)
#     return float(out.rx2("alpha")[0])


# def ipcw_intercept_and_slope_t0_stpm2(
#     model,
#     r_df,
#     t0_months: float,
#     event_col: str = "Event",
#     time_col: str = "Disease_Duration",
# ):
#     S_pred = predict_surv_at_t0(model, r_df, t0_months)
#     p_pred = 1.0 - np.asarray(S_pred, float)
#     p_pred = np.clip(p_pred, 1e-12, 1 - 1e-12)

#     time_np   = np.asarray(r_df.rx2(time_col), dtype=float)
#     status_np = np.asarray(r_df.rx2(event_col), dtype=int)

#     time_r   = FloatVector(time_np.tolist())
#     status_r = IntVector(status_np.tolist())
#     p_pred_r = FloatVector(p_pred.tolist())

#     out = _r_ipcw_recal_t0(time_r, status_r, p_pred_r, float(t0_months), True)
#     alpha = float(out.rx2("alpha")[0])
#     beta  = float(out.rx2("beta")[0])
#     return alpha, beta



# def ipcw_citl_from_pred_risk(
#     r_df,
#     p_pred: np.ndarray,
#     t0_months: float,
#     event_col: str = "Event",
#     time_col: str = "Disease_Duration",
#     weight_col: str | None = "weight",
# ) -> float:
#     p_pred = np.asarray(p_pred, float)
#     p_pred = np.clip(p_pred, 1e-12, 1 - 1e-12)

#     time_np   = np.asarray(r_df.rx2(time_col), dtype=float)
#     status_np = np.asarray(r_df.rx2(event_col), dtype=int)

#     time_r   = FloatVector(time_np.tolist())
#     status_r = IntVector(status_np.tolist())
#     p_pred_r = FloatVector(p_pred.tolist())

#     if weight_col is not None:
#         base_w_np = np.asarray(r_df.rx2(weight_col), dtype=float)
#         base_w_r = FloatVector(base_w_np.tolist())
#     else:
#         base_w_r = ro.NULL

#     out = _r_ipcw_recal_t0_w(time_r, status_r, p_pred_r, float(t0_months), base_w_r, False)
#     return float(out.rx2("alpha")[0])


# # ============================================================
# # Conversion helper
# # ============================================================ full
# def to_r_df(df: pd.DataFrame):
#     with localconverter(default_converter + pandas2ri.converter):
#         return pandas2ri.py2rpy(df)

# # ============================================================
# # LP / Score helpers
# # ============================================================
# def predict_pi_from_stpm2(model, r_df, vars_final: list[str]) -> np.ndarray:
#     """
#     Prognostic index PI(x) = x^T beta using ONLY covariate coefficients
#     that match vars_final exactly.
#     r_df must contain those columns as numeric.
#     """
#     coef_vec = ro.r["coef"](model)
#     coef_names = list(map(str, ro.r["names"](coef_vec)))
#     b = np.asarray(list(coef_vec), dtype=float)
#     coef_map = dict(zip(coef_names, b))

#     missing = [v for v in vars_final if v not in coef_map]
#     if missing:
#         raise ValueError(
#             f"vars_final not found in coef(model): {missing}\n"
#             f"First 40 coef names: {coef_names[:40]}"
#         )

#     # Extract X from r_df (as numpy)
#     Xcols = []
#     for v in vars_final:
#         Xcols.append(np.asarray(r_df.rx2(v), dtype=float))
#     X = np.column_stack(Xcols)  # n x p

#     beta = np.asarray([coef_map[v] for v in vars_final], dtype=float)
#     return X @ beta


# def _event_mask_from_r(r_df, event_col: str = "Event") -> np.ndarray:
#     """Extract Event vector from an R df and return boolean mask event==1."""
#     ev = np.asarray(r_df.rx2(event_col), dtype=float)
#     return (ev == 1)

# def choose_score_sign_from_lp(
#     lp: np.ndarray,
#     event_mask: np.ndarray,
#     weights: np.ndarray | None = None,
# ) -> int:
#     """
#     Decide whether score should be LP (+1) or -LP (-1)
#     so that weighted mean(score|event=1) > weighted mean(score|event=0).
#     """
#     lp = np.asarray(lp, float)
#     event_mask = np.asarray(event_mask, bool)

#     if weights is None:
#         m1 = float(np.mean(lp[event_mask])) if np.any(event_mask) else np.nan
#         m0 = float(np.mean(lp[~event_mask])) if np.any(~event_mask) else np.nan
#     else:
#         w = np.asarray(weights, float)
#         m1 = weighted_mean_np(lp[event_mask], w[event_mask]) if np.any(event_mask) else np.nan
#         m0 = weighted_mean_np(lp[~event_mask], w[~event_mask]) if np.any(~event_mask) else np.nan

#     if not np.isfinite(m1) or not np.isfinite(m0):
#         return +1

#     return +1 if (m1 > m0) else -1

# def risk_score_stpm2(
#     model,
#     r_df,
#     vars_final: list[str],
#     event_col: str = "Event",
#     weight_col: str | None = "weight",
#     sign: int | None = None,
#     return_details: bool = False,
# ):
#     """
#     Risk score based on PI(x)=x^T beta (covariates only, no baseline spline leakage).
#     Higher score => higher hazard (worse).
#     """
#     pi = predict_pi_from_stpm2(model, r_df, vars_final=vars_final)
#     event_mask = _event_mask_from_r(r_df, event_col=event_col)

#     w = None
#     if weight_col is not None:
#         try:
#             w = np.asarray(r_df.rx2(weight_col), dtype=float)
#         except Exception:
#             w = None

#     if sign is None:
#         sign = choose_score_sign_from_lp(pi, event_mask, weights=w)

#     score = sign * pi

#     if not return_details:
#         return score

#     if w is None:
#         m1_pi = float(np.mean(pi[event_mask])) if np.any(event_mask) else np.nan
#         m0_pi = float(np.mean(pi[~event_mask])) if np.any(~event_mask) else np.nan
#         m1_sc = float(np.mean(score[event_mask])) if np.any(event_mask) else np.nan
#         m0_sc = float(np.mean(score[~event_mask])) if np.any(~event_mask) else np.nan
#     else:
#         m1_pi = weighted_mean_np(pi[event_mask], w[event_mask]) if np.any(event_mask) else np.nan
#         m0_pi = weighted_mean_np(pi[~event_mask], w[~event_mask]) if np.any(~event_mask) else np.nan
#         m1_sc = weighted_mean_np(score[event_mask], w[event_mask]) if np.any(event_mask) else np.nan
#         m0_sc = weighted_mean_np(score[~event_mask], w[~event_mask]) if np.any(~event_mask) else np.nan

#     return score, {
#         "chosen_sign": int(sign),
#         "mean_pi_event1": m1_pi,
#         "mean_pi_event0": m0_pi,
#         "mean_score_event1": m1_sc,
#         "mean_score_event0": m0_sc,
#     }

# # ============================================================
# # Performance metrics (direction-safe)
# # ============================================================
# def cindex_from_risk_score(
#     r_df,
#     score: np.ndarray,
#     weight_col: str | None = "weight",
# ) -> float:
#     """
#     Harrell's C computed consistently.
#     survival::concordance assumes larger x => longer survival, so use -score.
#     """
#     r_tmp = R("transform")(r_df, score=FloatVector(np.asarray(score, float).tolist()))

#     if weight_col is None:
#         conc_obj = survival.concordance(
#             Formula("Surv(Disease_Duration, Event==1) ~ I(-score)"),
#             data=r_tmp
#         )
#     else:
#         conc_obj = survival.concordance(
#             Formula("Surv(Disease_Duration, Event==1) ~ I(-score)"),
#             data=r_tmp,
#             weights=r_tmp.rx2(weight_col)
#         )

#     return float(_r_conc_extract(conc_obj)[0])


# def slope_from_risk_score(
#     r_df,
#     score: np.ndarray,
#     weight_col: str | None = "weight",
# ) -> float:
#     r_tmp = R("transform")(
#         r_df,
#         score=FloatVector(np.asarray(score, float).tolist())
#     )

#     if weight_col is None:
#         cal = survival.coxph(
#             Formula("Surv(Disease_Duration, Event==1) ~ score"),
#             data=r_tmp
#         )
#     else:
#         cal = survival.coxph(
#             Formula(f"Surv(Disease_Duration, Event==1) ~ score"),
#             data=r_tmp,
#             weights=r_tmp.rx2(weight_col),
#             robust=True
#         )

#     return float(np.asarray(R["coef"](cal), dtype=float)[0])


# def weighted_mean_np(x: np.ndarray, w: np.ndarray | None = None) -> float:
#     x = np.asarray(x, float)
#     if w is None:
#         return float(np.mean(x))
#     w = np.asarray(w, float)
#     return float(np.sum(w * x) / np.sum(w))

# # ============================================================
# # CITL and survival-at-t0 (sign independent)
# # ============================================================
# def predict_surv_at_t0(model, r_df, t0_months: float) -> np.ndarray:
#     return np.asarray(_r_predict_surv_t0(model, r_df, float(t0_months)), dtype=float)

# def _logit(p: float) -> float:
#     p = float(np.clip(p, 1e-12, 1-1e-12))
#     return float(np.log(p/(1-p)))

# def km_surv_at_t0(r_df, t0_months: float, weight_col: str = "weight") -> float:
#     return float(_r_km_surv_t0_w(r_df, float(t0_months), weight_col)[0])

# def citl_at_t0(
#     model,
#     r_df,
#     t0_months: int,
#     weight_col: str | None = "weight",
# ) -> float:
#     """
#     CITL(t0) = logit(p_obs(t0)) - logit(weighted mean p_pred(t0))
#     """
#     S_pred = predict_surv_at_t0(model, r_df, t0_months)
#     p_pred = 1.0 - S_pred

#     if weight_col is not None:
#         try:
#             w = np.asarray(r_df.rx2(weight_col), dtype=float)
#         except Exception:
#             w = None
#     else:
#         w = None

#     p_bar = weighted_mean_np(p_pred, w)

#     S_km = km_surv_at_t0(r_df, t0_months, weight_col=weight_col if weight_col is not None else "weight")
#     if not np.isfinite(S_km):
#         return np.nan

#     p_obs = 1.0 - float(S_km)
#     return float(_logit(p_obs) - _logit(p_bar))


# def delta_logit_km_vs_meanpred_at_t0(model, r_df, t0_months: int) -> float:
#     # your existing citl_at_t0() exactly as is
#     return citl_at_t0(model, r_df, t0_months)


# def citl_ipcw_at_t0(model, r_df, t0_months: int) -> float:
#     return ipcw_calibration_intercept_t0_stpm2(model, r_df, t0_months)

# # ============================================================
# # Shrinkage helper (valid only when slope_corr > 0)
# # ============================================================

# def uniform_shrinkage_from_slope(slope_corr: float) -> float:
#     """
#     Uniform shrinkage factor s ≈ optimism-corrected calibration slope.

#     - If slope_corr < 1: overfitting -> shrink (s < 1)
#     - If slope_corr > 1: underfitting -> do not "anti-shrink" -> cap at 1
#     """
#     if not np.isfinite(slope_corr) or slope_corr <= 0:
#         return 1.0
#     return float(min(1.0, slope_corr))

# # ============================================================
# # Convenience: evaluate a fitted stpm2 model on a dataset
# # (apparent metrics; fully direction-safe)
# # ============================================================
# def evaluate_stpm2_apparent(
#     model,
#     r_df,
#     vars_final,
#     t0_months: int = 24,
#     event_col: str = "Event",
#     weight_col: str | None = "weight",
#     sign: int | None = None,
#     verbose: bool = True,
# ):
#     score, info = risk_score_stpm2(
#         model, r_df,
#         vars_final=vars_final,
#         event_col=event_col,
#         weight_col=weight_col,
#         sign=sign,
#         return_details=True
#     )

#     c = cindex_from_risk_score(r_df, score, weight_col=weight_col)
#     s = slope_from_risk_score(r_df, score, weight_col=weight_col)
#     citl_ipcw = ipcw_citl_from_pred_risk(
#         r_df,
#         p_pred=1.0 - predict_surv_at_t0(model, r_df, t0_months),
#         t0_months=t0_months,
#         event_col=event_col,
#         weight_col=weight_col,
#     )
#     citl_simple = citl_at_t0(model, r_df, t0_months, weight_col=weight_col)

#     if verbose:
#         print("Score sign chosen:", info["chosen_sign"])
#         print("Mean PI event=1 / event=0:", info["mean_pi_event1"], info["mean_pi_event0"])
#         print("Mean score event=1 / event=0:", info["mean_score_event1"], info["mean_score_event0"])
#         print("C-index:", c)
#         print("Slope:", s)
#         print("CITL IPCW:", citl_ipcw)
#         print("CITL simple:", citl_simple)

#     return {
#         "cindex": c,
#         "slope": s,
#         "citl_ipcw": citl_ipcw,
#         "citl_simple": citl_simple,
#         "score_sign": info["chosen_sign"],
#         "details": info
#     }

# # ============================================================
# # (Optional) Calibration bins utility (KM risk with CI)
# # ============================================================

# def calibration_bins_km(r_df, pred_risk: np.ndarray, t0_months: int, n_bins: int = 5) -> pd.DataFrame:
#     df = pd.DataFrame({"pred": np.asarray(pred_risk, float)})
#     df["bin"] = pd.qcut(df["pred"], q=n_bins, duplicates="drop")

#     rows = []
#     for b in df["bin"].cat.categories:
#         mask = (df["bin"] == b).to_numpy()
#         idx0 = np.where(mask)[0]
#         if len(idx0) == 0:
#             continue
#         r_sub = r_df.rx(IntVector((idx0 + 1).tolist()), True)
#         risk, lo, hi = km_risk_ci_at_t0(r_sub, t0_months)
#         if not np.isfinite(risk):
#             continue
#         rows.append({
#             "bin": str(b),
#             "n": int(mask.sum()),
#             "mean_pred": float(df.loc[mask, "pred"].mean()),
#             "obs_km": float(risk),
#             "obs_lo": float(lo),
#             "obs_hi": float(hi),
#         })
#     return pd.DataFrame(rows)


# def calibration_bins_km_robust(
#     r_df,
#     pred_risk: np.ndarray,
#     t0_months: int,
#     n_bins: int = 5,
#     weight_col: str = "weight",
# ) -> pd.DataFrame:
#     pr = np.asarray(pred_risk, float)
#     df = pd.DataFrame({"pred": pr})
#     df = df[np.isfinite(df["pred"])].copy()
#     if df.empty:
#         return pd.DataFrame()

#     try:
#         df["bin"] = pd.qcut(df["pred"], q=n_bins, duplicates="drop")
#         if df["bin"].isna().all() or len(df["bin"].cat.categories) < 2:
#             raise ValueError("qcut collapsed bins")
#     except Exception:
#         r = df["pred"].rank(method="average")
#         df["bin"] = pd.qcut(r, q=min(n_bins, df.shape[0]), duplicates="drop")

#     rows = []
#     for b in df["bin"].cat.categories:
#         mask = (df["bin"] == b).to_numpy()
#         idx0 = np.where(mask)[0]
#         if len(idx0) == 0:
#             continue

#         r_sub = r_df.rx(IntVector((idx0 + 1).tolist()), True)
#         risk, lo, hi = km_risk_ci_at_t0(r_sub, t0_months, weight_col=weight_col)
#         if not np.isfinite(risk):
#             continue

#         w_sub = np.asarray(r_sub.rx2(weight_col), dtype=float)
#         pred_sub = df.loc[mask, "pred"].to_numpy(float)
#         mean_pred = weighted_mean_np(pred_sub, w_sub)

#         rows.append({
#             "bin": str(b),
#             "n": int(mask.sum()),
#             "mean_pred": float(mean_pred),
#             "obs_km": float(risk),
#             "obs_lo": float(lo),
#             "obs_hi": float(hi),
#         })

#     return pd.DataFrame(rows)


# # ============================================================
# # QUICK USAGE EXAMPLE (after you fit model_best_full)
# # ============================================================
# # with localconverter(default_converter + pandas2ri.converter):
# #     r_full = pandas2ri.py2rpy(dev_df)
# #
# # out = evaluate_stpm2_apparent(model_best_full, r_full, t0_months=24, verbose=True)
# # ============================================================

# # ============================================================
# # FULL-STRATEGY BOOTSTRAP VALIDATION (CORRECTED)
# # ------------------------------------------------------------
# # - Repeats the *entire* modelling strategy inside each bootstrap:
# #     * backward AIC selection (with forced include/exclude)
# #     * baseline spline df selection (over df_candidates)
# # - Uses automatic direction detection for the risk score:
# #     * score = s * LP, where s ∈ {+1, -1} chosen so that
# #           mean(score | Event=1) > mean(score | Event=0)
# # - FIXES SIGN CONSISTENCY:
# #     * sign is chosen ON THE BOOTSTRAP SAMPLE (in-sample)
# #     * the SAME sign is used when evaluating OUT-OF-BOOTSTRAP
# #       on the original full dataset (no re-detection)
# # - Computes optimism-corrected:
# #     * C-index (via concordance with I(-score))
# #     * Calibration slope (Cox on score)
# #     * CITL at t0 (from predicted survival & KM)
# #
# # Requires your previously pasted helpers:
# #   - backward_elimination_aic (SelectionResult)
# #   - to_r_df
# #   - risk_score_stpm2 (with sign optional)
# #   - cindex_from_risk_score
# #   - slope_from_risk_score
# #   - citl_at_t0
# #
# # And assumes: rstpm2, survival, stats are available.
# # ============================================================



# R = ro.r

# @dataclass
# class FullStrategyValidationResult:
#     best_vars_full: list[str]
#     best_spline_df_full: int
#     best_aic_full: float
#     best_formula_full: str

#     cindex_app: float
#     slope_app: float
#     citl_app: float

#     cindex_corr: float
#     slope_corr: float
#     citl_corr: float

#     cindex_ci: tuple[float, float]
#     slope_ci: tuple[float, float]
#     citl_ci: tuple[float, float]

#     n_boot_success: int
#     n_boot_fail: int

# def _model_signature(vars_list: list[str], spline_df: int) -> str:
#     key = f"df={spline_df}|" + "+".join(vars_list)
#     return hashlib.md5(key.encode("utf-8")).hexdigest()

# def backward_elimination_aic(
#     df_python: pd.DataFrame,
#     covariates: list[str],
#     df_candidates: list[int],
#     hard_include: list[str],
#     hard_exclude: list[str] | None = None,
#     weight_col: str | None = "weight",
#     robust: bool = True,
# ) -> SelectionResult:
#     hard_exclude = list(hard_exclude or [])

#     start_vars = [v for v in covariates if v not in set(hard_exclude)]
#     for v in hard_include:
#         if v not in start_vars and v not in set(hard_exclude):
#             start_vars.append(v)

#     forced = set(hard_include)
#     r_df = _to_r_df(df_python.reset_index(drop=True))
#     current_vars = start_vars[:]
#     best: SelectionResult | None = None

#     def better(a: SelectionResult, b: SelectionResult | None) -> bool:
#         if b is None:
#             return True
#         return a.aic < b.aic - 1e-12

#     while len(current_vars) >= 1:
#         candidates: list[SelectionResult] = []

#         for ddf in df_candidates:
#             _, aic, fml = fit_stpm2_and_aic(
#                 r_df, current_vars, ddf,
#                 weight_col=weight_col,
#                 robust=robust
#             )
#             candidates.append(SelectionResult(current_vars[:], ddf, aic, fml))

#         removable = [v for v in current_vars if v not in forced]
#         for var in removable:
#             test_vars = [v for v in current_vars if v != var]
#             if not test_vars:
#                 continue
#             for ddf in df_candidates:
#                 _, aic, fml = fit_stpm2_and_aic(
#                     r_df, test_vars, ddf,
#                     weight_col=weight_col,
#                     robust=robust
#                 )
#                 candidates.append(SelectionResult(test_vars[:], ddf, aic, fml))

#         cand_best = None
#         for c in candidates:
#             if better(c, cand_best):
#                 cand_best = c

#         if cand_best is None:
#             break
#         if best is not None and cand_best.aic >= best.aic - 1e-12:
#             break

#         best = cand_best
#         current_vars = best.vars_final[:]
#         if all(v in forced for v in current_vars):
#             break

#     if best is None:
#         forced_list = [v for v in hard_include if v not in set(hard_exclude)]
#         ddf = df_candidates[0]
#         _, aic, fml = fit_stpm2_and_aic(
#             r_df, forced_list, ddf,
#             weight_col=weight_col,
#             robust=robust
#         )
#         best = SelectionResult(forced_list, ddf, aic, fml)

#     return best

# def full_strategy_bootstrap_validation_corrected(
#     df_full: pd.DataFrame,
#     covariates: list[str],
#     df_candidates: list[int],
#     hard_include: list[str],
#     hard_exclude: list[str] | None,
#     t0_months: int,
#     n_boot: int,
#     random_state: int | None = None,
#     event_col: str = "Event",
#     weight_col: str | None = "weight",
#     robust: bool = True,
#     verbose: bool = True,
# ):
#     hard_exclude = list(hard_exclude or [])
#     rng = np.random.default_rng(random_state)

#     df_full = df_full.reset_index(drop=True).copy()
#     r_full = to_r_df(df_full)

#     sel_full = backward_elimination_aic(
#         df_python=df_full,
#         covariates=covariates,
#         df_candidates=df_candidates,
#         hard_include=hard_include,
#         hard_exclude=hard_exclude,
#         weight_col=weight_col,
#         robust=robust,
#     )

#     if weight_col is not None:
#         model_best_full = rstpm2.stpm2(
#             Formula(sel_full.formula),
#             data=r_full,
#             df=sel_full.spline_df,
#             weights=r_full.rx2(weight_col),
#             robust=robust
#         )
#     else:
#         model_best_full = rstpm2.stpm2(
#             Formula(sel_full.formula),
#             data=r_full,
#             df=sel_full.spline_df
#         )

#     score_full, info_full = risk_score_stpm2(
#         model_best_full, r_full,
#         vars_final=sel_full.vars_final,
#         event_col=event_col,
#         weight_col=weight_col,
#         sign=None,
#         return_details=True
#     )

#     c_app = cindex_from_risk_score(r_full, score_full, weight_col=weight_col)
#     s_app = slope_from_risk_score(r_full, score_full, weight_col=weight_col)
#     citl_app = citl_ipcw_at_t0(model_best_full, r_full, t0_months)

#     if verbose:
#         print("\n[Apparent] score sign on FULL data:", info_full["chosen_sign"])
#         print("[Apparent] mean PI event1/event0:", info_full["mean_pi_event1"], info_full["mean_pi_event0"])
#         print("[Apparent] mean score event1/event0:", info_full["mean_score_event1"], info_full["mean_score_event0"])

#     n = len(df_full)
#     c_diffs, s_diffs, citl_diffs = [], [], []
#     c_out_list, s_out_list, citl_out_list = [], [], []
#     boot_coefs = []

#     incl_counts = Counter()
#     df_counts = Counter()
#     model_counts = Counter()

#     n_success = 0
#     n_fail = 0

#     for b in range(n_boot):
#         idx = rng.integers(0, n, size=n)
#         df_boot = df_full.iloc[idx].reset_index(drop=True)

#         try:
#             sel_boot = backward_elimination_aic(
#                 df_python=df_boot,
#                 covariates=covariates,
#                 df_candidates=df_candidates,
#                 hard_include=hard_include,
#                 hard_exclude=hard_exclude,
#                 weight_col=weight_col,
#                 robust=robust,
#             )

#             incl_counts.update(sel_boot.vars_final)
#             df_counts.update([sel_boot.spline_df])
#             model_counts.update([_model_signature(sel_boot.vars_final, sel_boot.spline_df)])

#             r_boot = to_r_df(df_boot)

#             if weight_col is not None:
#                 model_boot = rstpm2.stpm2(
#                     Formula(sel_boot.formula),
#                     data=r_boot,
#                     df=sel_boot.spline_df,
#                     weights=r_boot.rx2(weight_col),
#                     robust=robust
#                 )
#             else:
#                 model_boot = rstpm2.stpm2(
#                     Formula(sel_boot.formula),
#                     data=r_boot,
#                     df=sel_boot.spline_df
#                 )

#             if (
#                 sel_boot.vars_final == sel_full.vars_final and
#                 sel_boot.spline_df == sel_full.spline_df
#             ):
#                 bvec = np.asarray(list(ro.r["coef"](model_boot)), dtype=float)
#                 boot_coefs.append(bvec)

#             score_in, info_in = risk_score_stpm2(
#                 model_boot, r_boot,
#                 vars_final=sel_boot.vars_final,
#                 event_col=event_col,
#                 weight_col=weight_col,
#                 sign=None,
#                 return_details=True
#             )
#             sign_boot = info_in["chosen_sign"]

#             c_in = cindex_from_risk_score(r_boot, score_in, weight_col=weight_col)
#             s_in = slope_from_risk_score(r_boot, score_in, weight_col=weight_col)
#             citl_in = citl_ipcw_at_t0(model_boot, r_boot, t0_months)

#             score_out = risk_score_stpm2(
#                 model_boot, r_full,
#                 vars_final=sel_boot.vars_final,
#                 event_col=event_col,
#                 weight_col=weight_col,
#                 sign=sign_boot,
#                 return_details=False
#             )

#             c_out = cindex_from_risk_score(r_full, score_out, weight_col=weight_col)
#             s_out = slope_from_risk_score(r_full, score_out, weight_col=weight_col)
#             citl_out = citl_ipcw_at_t0(model_boot, r_full, t0_months)

#             c_diffs.append(c_in - c_out)
#             s_diffs.append(s_in - s_out)
#             citl_diffs.append(citl_in - citl_out)

#             c_out_list.append(c_out)
#             s_out_list.append(s_out)
#             citl_out_list.append(citl_out)

#             n_success += 1

#         except Exception:
#             n_fail += 1



# # ============================================================
# # ADD-ON: SUMMARY TABLES (CI in separate column) + COEFFICIENT TABLE
# #         (non-updated vs updated/shrunk) + CALIBRATION PLOTS
# #
# # Assumes you already ran the previous "RUN ALL" cell and you have:
# #   res_strategy, model_best_full, res_ext, shrinkage
# #
# # Notes:
# # - "Updated coefficients" here = uniform-shrinkage updated coefficients (beta * shrinkage).
# #   Baseline updating at t0 changes ONLY the baseline (absolute risk), not betas.
# # - Coefficient CIs:
# #     * Non-updated: Wald CI from stpm2 vcov (fast, standard)
# #     * Updated/shrunk: Wald CI scaled by shrinkage (approximation, commonly used)
# #
# # - Plots: calibration (frozen vs baseline-updated) with KM Greenwood CI error bars.
# # ============================================================


# # ----------------------------
# # Helper: format CI into separate column
# # ----------------------------
# def ci_str(lo, hi, digits=3):
#     if lo is None or hi is None or (not np.isfinite(lo)) or (not np.isfinite(hi)):
#         return ""
#     return f"[{lo:.{digits}f}, {hi:.{digits}f}]"

# # ----------------------------
# # Summary table for INTERNAL (strategy) results
# # CI is a separate column
# # ----------------------------
# def make_strategy_summary_table(res_strategy, t0_months: int) -> pd.DataFrame:
#     rows = []
#     def add(metric, value=None, lo=None, hi=None, digits=3):
#         if isinstance(value, (float, int, np.floating)) and value is not None and np.isfinite(value):
#             v = f"{float(value):.{digits}f}"
#         else:
#             v = value
#         rows.append({"Metric": metric, "Value": v, "95% CI": ci_str(lo, hi, digits=digits)})

#     add("Best model on full data (strategy)", ", ".join(res_strategy.best_vars_full))
#     add("Best spline df on full data", res_strategy.best_spline_df_full, digits=0)
#     add("AIC (best on full data)", res_strategy.best_aic_full)

#     add("C-index (apparent, best model)", res_strategy.cindex_app)
#     add("C-index (optimism-corrected, strategy)", res_strategy.cindex_corr,
#         res_strategy.cindex_ci[0], res_strategy.cindex_ci[1])

#     add("Calibration slope (apparent, best model)", res_strategy.slope_app)
#     add("Calibration slope (optimism-corrected, strategy)", res_strategy.slope_corr,
#         res_strategy.slope_ci[0], res_strategy.slope_ci[1])

#     add(f"CITL at {t0_months} months (apparent, best model)", res_strategy.citl_app)
#     add(f"CITL at {t0_months} months (optimism-corrected, strategy)", res_strategy.citl_corr,
#         res_strategy.citl_ci[0], res_strategy.citl_ci[1])

#     add("Bootstrap reps (success)", res_strategy.n_boot_success, digits=0)
#     add("Bootstrap reps (fail)", res_strategy.n_boot_fail, digits=0)

#     return pd.DataFrame(rows, columns=["Metric", "Value", "95% CI"])

# # ----------------------------
# # Summary table for EXTERNAL results (CI column included where applicable)
# # For external: we typically don't have CI for c-index/slope here (unless bootstrapped externally).
# # But calibration bins already contain KM CIs; we keep those in the calibration tables.
# # ----------------------------
# def make_external_summary_table(res_ext, t0_months: int) -> pd.DataFrame:
#     rows = []
#     def add(metric, value=None, lo=None, hi=None, digits=3):
#         if isinstance(value, (float, int, np.floating)) and value is not None and np.isfinite(value):
#             v = f"{float(value):.{digits}f}"
#         else:
#             v = value
#         rows.append({"Metric": metric, "Value": v, "95% CI": ci_str(lo, hi, digits=digits)})

#     add("External time horizon (months)", t0_months, digits=0)
#     add("Model variables", ", ".join(res_ext.get("vars_final", [])))
#     add("Shrinkage factor (uniform)", res_ext.get("shrinkage", np.nan))

#     add("C-index (external)", res_ext.get("cindex_external", np.nan))
#     add("Calibration slope (external)", res_ext.get("slope_external", np.nan))

#     add("Observed KM survival at t0", res_ext.get("S_obs_km", np.nan))
#     add("Observed KM risk at t0", res_ext.get("obs_risk_km", np.nan))

#     add("Mean predicted risk at t0 (frozen)", res_ext.get("mean_pred_risk_frozen", np.nan))
#     add("Baseline update factor", res_ext.get("baseline_factor", np.nan))
#     add("Mean predicted risk at t0 (baseline-updated)", res_ext.get("mean_pred_risk_updated", np.nan))

#     return pd.DataFrame(rows, columns=["Metric", "Value", "95% CI"])

# # ----------------------------
# # Coefficient table: non-updated (apparent) + updated (shrunken)
# # Wald CI from stpm2 vcov, then scale by shrinkage for the updated CI
# # ----------------------------
# def coef_table_apparent_and_shrunk(model_best_full, vars_final: list[str], shrinkage: float) -> pd.DataFrame:
#     # coefficients and vcov from R
#     coef_vec = ro.r["coef"](model_best_full)
#     vc = ro.r["vcov"](model_best_full)

#     names = list(ro.r["names"](coef_vec))
#     b = np.asarray(list(coef_vec), dtype=float)

#     # Map name->index
#     idx = {str(n): i for i, n in enumerate(names)}

#     # Convert vcov to numpy
#     vc_np = np.asarray(vc, dtype=float)

#     rows = []
#     for v in vars_final:
#         if v not in idx:
#             # if you ever move binaries to factors, you'd handle mapping here
#             rows.append({
#                 "Variable": v,
#                 "Beta (apparent)": "",
#                 "95% CI (apparent)": "",
#                 "Beta (updated/shrunk)": "",
#                 "95% CI (updated/shrunk)": "",
#             })
#             continue

#         i = idx[v]
#         beta = float(b[i])
#         se = float(np.sqrt(vc_np[i, i])) if np.isfinite(vc_np[i, i]) else np.nan

#         lo = beta - 1.96 * se
#         hi = beta + 1.96 * se

#         beta_s = float(beta * shrinkage)
#         lo_s = float(lo * shrinkage)
#         hi_s = float(hi * shrinkage)

#         rows.append({
#             "Variable": v,
#             "Beta (apparent)": f"{beta:.3f}",
#             "95% CI (apparent)": ci_str(lo, hi, digits=3),
#             "Beta (updated/shrunk)": f"{beta_s:.3f}",
#             "95% CI (updated/shrunk)": ci_str(lo_s, hi_s, digits=3),
#         })

#     return pd.DataFrame(rows)



# def coef_table_apparent_and_shrunk_2(
#     model_best_full,
#     vars_final: list[str],
#     shrinkage: float,
#     digits_beta: int = 3,
#     digits_hr: int = 3,
# ) -> pd.DataFrame:
#     """
#     Build coefficient table for selected covariates from an stpm2 model:
#       - log(HR) (beta) apparent with 95% CI
#       - HR apparent with 95% CI
#       - log(HR) updated/shrunk with 95% CI
#       - HR updated/shrunk with 95% CI

#     Assumes covariate term names in coef(model) match vars_final exactly
#     (i.e., numeric 0/1 binaries, not factor-expanded names).
#     """

#     def ci_str(lo, hi, digits=3):
#         if lo is None or hi is None or (not np.isfinite(lo)) or (not np.isfinite(hi)):
#             return ""
#         return f"[{lo:.{digits}f}, {hi:.{digits}f}]"

#     def fmt(x, d):
#         return "" if (x is None or (isinstance(x, float) and not np.isfinite(x))) else f"{float(x):.{d}f}"

#     # coefficients and vcov from R
#     coef_vec = ro.r["coef"](model_best_full)
#     vc = ro.r["vcov"](model_best_full)

#     names = list(ro.r["names"](coef_vec))
#     b = np.asarray(list(coef_vec), dtype=float)

#     # Map name->index
#     idx = {str(n): i for i, n in enumerate(names)}

#     # Convert vcov to numpy
#     vc_np = np.asarray(vc, dtype=float)

#     rows = []
#     for v in vars_final:
#         if v not in idx:
#             rows.append({
#                 "Variable": v,

#                 "log(HR) apparent": "",
#                 "95% CI log(HR) apparent": "",
#                 "HR apparent": "",
#                 "95% CI HR apparent": "",

#                 "log(HR) shrunk": "",
#                 "95% CI log(HR) shrunk": "",
#                 "HR shrunk": "",
#                 "95% CI HR shrunk": "",
#             })
#             continue

#         i = idx[v]
#         beta = float(b[i])
#         var_i = float(vc_np[i, i]) if np.isfinite(vc_np[i, i]) else np.nan
#         se = float(np.sqrt(var_i)) if np.isfinite(var_i) and var_i >= 0 else np.nan

#         # Apparent CI on log-HR scale
#         lo = beta - 1.96 * se if np.isfinite(se) else np.nan
#         hi = beta + 1.96 * se if np.isfinite(se) else np.nan

#         # HR scale (exp)
#         hr = float(np.exp(beta))
#         hr_lo = float(np.exp(lo)) if np.isfinite(lo) else np.nan
#         hr_hi = float(np.exp(hi)) if np.isfinite(hi) else np.nan

#         # Shrunk (uniform shrinkage on beta & its CI on log scale)
#         beta_s = float(beta * shrinkage)
#         lo_s = float(lo * shrinkage) if np.isfinite(lo) else np.nan
#         hi_s = float(hi * shrinkage) if np.isfinite(hi) else np.nan

#         hr_s = float(np.exp(beta_s))
#         hr_s_lo = float(np.exp(lo_s)) if np.isfinite(lo_s) else np.nan
#         hr_s_hi = float(np.exp(hi_s)) if np.isfinite(hi_s) else np.nan

#         rows.append({
#             "Variable": v,

#             "log(HR) apparent": fmt(beta, digits_beta),
#             "95% CI log(HR) apparent": ci_str(lo, hi, digits=digits_beta),
#             "HR apparent": fmt(hr, digits_hr),
#             "95% CI HR apparent": ci_str(hr_lo, hr_hi, digits=digits_hr),

#             "log(HR) shrunk": fmt(beta_s, digits_beta),
#             "95% CI log(HR) shrunk": ci_str(lo_s, hi_s, digits=digits_beta),
#             "HR shrunk": fmt(hr_s, digits_hr),
#             "95% CI HR shrunk": ci_str(hr_s_lo, hr_s_hi, digits=digits_hr),
#         })

#     return pd.DataFrame(rows)

# # ----------------------------
# # Calibration plot (frozen or baseline-updated)
# # cal_df must have numeric columns: mean_pred, obs_km, obs_lo, obs_hi
# # (these are produced by your external validation block)
# # ----------------------------
# def plot_calibration_table(
#     cal_df: pd.DataFrame,
#     title: str,
#     percent: bool = True,
#     show_45deg: bool = True,
#     ylim: tuple[float, float] | None = None,
# ):
#     if cal_df is None or len(cal_df) == 0:
#         raise ValueError("Calibration table is empty.")

#     # Ensure numeric
#     x = cal_df["mean_pred"].to_numpy(dtype=float)
#     y = cal_df["obs_km"].to_numpy(dtype=float)
#     lo = cal_df["obs_lo"].to_numpy(dtype=float)
#     hi = cal_df["obs_hi"].to_numpy(dtype=float)

#     ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(lo) & np.isfinite(hi)
#     x, y, lo, hi = x[ok], y[ok], lo[ok], hi[ok]

#     yerr_low = np.maximum(y - lo, 0.0)
#     yerr_high = np.maximum(hi - y, 0.0)

#     fig, ax = plt.subplots(figsize=(5, 5))
#     ax.errorbar(x, y, yerr=np.vstack([yerr_low, yerr_high]), fmt="o", capsize=3)

#     maxv = float(np.nanmax(np.r_[x, y, hi])) if len(x) else 1.0
#     maxv = max(maxv, 1e-6)

#     if show_45deg:
#         ax.plot([0, maxv], [0, maxv], "--")

#     ax.set_xlim(0, maxv)
#     if ylim is None:
#         ax.set_ylim(0, maxv)
#     else:
#         ax.set_ylim(ylim[0], ylim[1])

#     ax.set_xlabel("Predicted risk")
#     ax.set_ylabel("Observed risk")
#     ax.set_title(title)

#     if percent:
#         fmt = FuncFormatter(lambda v, p: f"{100*v:.0f}%")
#         ax.xaxis.set_major_formatter(fmt)
#         ax.yaxis.set_major_formatter(fmt)

#     ax.grid(alpha=0.3)
#     plt.tight_layout()
#     return fig, ax

# # ============================================================
# # ========================= OUTPUTS ===========================
# # ============================================================

# # Optional: If you want "clean" calibration tables with CI in a separate column for slides:
# def tidy_calibration_table(cal_df: pd.DataFrame) -> pd.DataFrame:
#     out = cal_df.copy()
#     out["Observed 95% CI"] = out.apply(lambda r: ci_str(float(r["obs_lo"]), float(r["obs_hi"]), digits=3), axis=1)
#     out["mean_pred"] = out["mean_pred"].map(lambda v: f"{float(v):.3f}")
#     out["obs_km"] = out["obs_km"].map(lambda v: f"{float(v):.3f}")
#     out = out[["bin", "n", "mean_pred", "obs_km", "Observed 95% CI"]]
#     out.columns = ["Bin", "N", "Mean predicted risk", "Observed risk (KM)", "Observed 95% CI"]
#     return out

# def tidy_calibration_df(cal_df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Ensures numeric columns are numeric and drops non-finite rows.
#     Expected columns: mean_pred, obs_km, obs_lo, obs_hi
#     """
#     df = cal_df.copy()
#     for c in ["mean_pred", "obs_km", "obs_lo", "obs_hi"]:
#         df[c] = pd.to_numeric(df[c], errors="coerce")
#     df = df.dropna(subset=["mean_pred", "obs_km", "obs_lo", "obs_hi"]).reset_index(drop=True)
#     return df



# # ============================================================
# # EXTERNAL VALIDATION (CORRECTED + SIGN CONSISTENCY)
# # ------------------------------------------------------------
# # Uses the SAME risk-score sign convention determined on the
# # DEVELOPMENT data for the final fitted model, and does NOT
# # re-detect sign on the external dataset.
# #
# # - Discrimination: Harrell C using I(-score) (direction-safe)
# # - Calibration slope: Cox on score (higher score = higher hazard)
# # - Absolute risk at t0: from predict(type="surv") (sign independent)
# # - Baseline recalibration at t0: S_updated = S_pred ^ factor
# # - Calibration-by-bins: KM risk w/ Greenwood CI within bins
# #
# # Requires helpers already defined:
# #   - to_r_df
# #   - risk_score_stpm2
# #   - cindex_from_risk_score
# #   - slope_from_risk_score
# #   - predict_surv_at_t0
# #   - km_surv_at_t0
# #   - calibration_bins_km
# #   - uniform_shrinkage_from_slope (or your shrinkage helper)
# #
# # And assumes: survival is available (R), rstpm2 model fitted.
# # ============================================================

# def external_validate_stpm2_corrected(
#     model_final,
#     dev_r_df,
#     ext_df: pd.DataFrame,
#     vars_final: list[str],
#     t0_months: int,
#     shrinkage: float = 1.0,
#     n_bins: int = 5,
#     baseline_update: bool = True,
#     event_col: str = "Event",
#     weight_col: str | None = "weight",
#     verbose: bool = True,
# ):
#     # -----------------------------
#     # 0) Ensure ext has needed cols
#     # -----------------------------
#     ext_df = ext_df.reset_index(drop=True).copy()

#     needed = ["Disease_Duration", event_col] + list(vars_final)
#     if weight_col is not None and weight_col in ext_df.columns:
#         needed = needed + [weight_col]

#     missing = [c for c in needed if c not in ext_df.columns]
#     if missing:
#         raise ValueError(f"External df missing columns: {missing}")

#     r_ext = to_r_df(ext_df[needed])

#     w_ext = np.asarray(r_ext.rx2(weight_col), dtype=float) if (weight_col is not None and weight_col in ext_df.columns) else None

#     # ---------------------------------------------------------
#     # 1) Determine score sign ON DEVELOPMENT DATA (once)
#     # ---------------------------------------------------------
#     _, info_dev = risk_score_stpm2(
#         model_final, dev_r_df,
#         vars_final=vars_final,
#         event_col=event_col,
#         sign=None,
#         return_details=True
#     )
#     dev_sign = info_dev["chosen_sign"]

#     if verbose:
#         print("\n[External] Using development-derived score sign:", dev_sign)
#         print("[Dev sign check] mean score event1/event0:",
#               info_dev["mean_score_event1"], info_dev["mean_score_event0"])

#     # ---------------------------------------------------------
#     # 2) Compute external risk score using FIXED sign
#     # ---------------------------------------------------------
#     score_ext = risk_score_stpm2(
#         model_final, r_ext,
#         vars_final=vars_final,
#         event_col=event_col,
#         sign=dev_sign,
#         return_details=False
#     )
#     score_ext_shrunk = np.asarray(score_ext, float) * float(shrinkage)

#     cindex_ext = cindex_from_risk_score(r_ext, score_ext_shrunk)
#     slope_ext  = slope_from_risk_score(r_ext, score_ext_shrunk)

#     if verbose:
#         ev = np.asarray(r_ext.rx2(event_col), dtype=float) == 1
#         if np.any(ev) and np.any(~ev):
#             print("[Ext check] mean score(event=1) vs score(event=0):",
#                   float(np.mean(score_ext_shrunk[ev])),
#                   float(np.mean(score_ext_shrunk[~ev])))

#     # ---------------------------------------------------------
#     # 3) Absolute risk at t0 (frozen baseline)
#     # ---------------------------------------------------------
#     S_pred = predict_surv_at_t0(model_final, r_ext, t0_months)  # sign independent
#     risk_frozen = 1.0 - S_pred

#     # Observed KM at t0 in external (keep for reporting)
#     S_obs = km_surv_at_t0(r_ext, t0_months, weight_col="weight")

#     # ---------------------------------------------------------
#     # 4) Baseline update at t0 (survival-scale recalibration)
#     # ---------------------------------------------------------
#     if not np.isfinite(S_obs):
#         factor = np.nan
#         risk_updated = risk_frozen.copy()
#     else:
#         S_obs_c = float(np.clip(S_obs, 1e-12, 1 - 1e-12))
#         S_bar_c = float(np.clip(np.mean(S_pred), 1e-12, 1 - 1e-12))

#         alpha = np.log(-np.log(S_obs_c)) - np.log(-np.log(S_bar_c))
#         factor = float(np.exp(alpha))

#         if baseline_update:
#             S_upd = np.power(S_pred, factor)
#             risk_updated = 1.0 - S_upd
#         else:
#             risk_updated = risk_frozen.copy()

#     w_ext = np.asarray(r_ext.rx2("weight"), dtype=float) if "weight" in ext_df.columns else None

#     mean_pred_risk_frozen = weighted_mean_np(risk_frozen, w_ext)
#     mean_pred_risk_updated = weighted_mean_np(risk_updated, w_ext)
    
#     # ---------------------------------------------------------
#     # 4b) External CITL at t0 (STANDARD via IPCW logistic recalibration)
#     # ---------------------------------------------------------
#     citl_frozen_ipcw = ipcw_citl_from_pred_risk(
#         r_ext, p_pred=risk_frozen, t0_months=t0_months, event_col=event_col, weight_col="weight"
#     )
#     citl_updated_ipcw = ipcw_citl_from_pred_risk(
#         r_ext, p_pred=risk_updated, t0_months=t0_months, event_col=event_col, weight_col="weight"
#     )

#     # ---------------------------------------------------------
#     # 5) Calibration-by-bins tables (KM risk + CI per bin)
#     # ---------------------------------------------------------
#     cal_frozen  = calibration_bins_km_robust(r_ext, risk_frozen,  t0_months, n_bins=n_bins, weight_col="weight")
#     cal_updated = calibration_bins_km_robust(r_ext, risk_updated, t0_months, n_bins=n_bins, weight_col="weight")

    

#     return {
#         "t0_months": int(t0_months),
#         "vars_final": list(vars_final),
#         "shrinkage": float(shrinkage),
#         "dev_score_sign": int(dev_sign),

#         "cindex_external": float(cindex_ext),
#         "slope_external": float(slope_ext),

#         "S_obs_km": float(S_obs) if np.isfinite(S_obs) else np.nan,
#         "obs_risk_km": float(1.0 - S_obs) if np.isfinite(S_obs) else np.nan,

#         "mean_pred_risk_frozen": float(mean_pred_risk_frozen),
#         "citl_external_frozen": float(citl_frozen_ipcw),

#         "baseline_factor": float(factor) if np.isfinite(factor) else np.nan,
#         "mean_pred_risk_updated": float(mean_pred_risk_updated),
#         "citl_external_updated": float(citl_updated_ipcw),

#         "calibration_table_frozen": cal_frozen,
#         "calibration_table_updated": cal_updated,
#     }



# # ============================================================
# # External bootstrap CIs (compact, clean) — NO refitting
# # ------------------------------------------------------------
# # Produces percentile 95% CIs for:
# #   - C-index (external)
# #   - Calibration slope (external)
# #   - CITL at t0: frozen + baseline-updated
# #   - Mean predicted risk at t0: frozen + updated
# #   - Observed KM risk at t0
# #
# # Keeps model_fixed constant (no refit). Recomputes:
# #   - score (using dev-derived sign), then cindex/slope
# #   - S_pred(t0) -> risk_frozen
# #   - KM S_obs(t0) -> p_obs
# #   - baseline factor -> risk_updated
# #   - CITL_frozen/CITL_updated
# #
# # Requirements (from your helpers):
# #   - to_r_df
# #   - risk_score_stpm2 (sign can be fixed)
# #   - cindex_from_risk_score
# #   - slope_from_risk_score
# #   - predict_surv_at_t0
# #   - km_surv_at_t0
# #
# # NOTE: This bootstraps individuals in the external dataset.
# # ============================================================


# def external_bootstrap_cis(
#     model_final,
#     dev_r_df,
#     ext_df: pd.DataFrame,
#     vars_final: list[str],
#     t0_months: int,
#     shrinkage: float = 1.0,
#     baseline_update: bool = True,
#     n_boot: int = 500,
#     random_state: int | None = 123,
#     event_col: str = "Event",
#     weight_col: str | None = "weight",
#     return_replicates: bool = False,
# ):
#     def logit(p: float) -> float:
#         p = float(np.clip(p, 1e-12, 1 - 1e-12))
#         return float(np.log(p/(1-p)))

#     def citl(p_obs: float, mean_pred: float) -> float:
#         return float(logit(p_obs) - logit(mean_pred))

#     # --- determine sign ON DEVELOPMENT (fixed for all external bootstraps) ---
    
#     _, info_dev = risk_score_stpm2(
#         model_final, dev_r_df,
#         vars_final=vars_final,
#         event_col=event_col,
#         sign=None,
#         return_details=True
#     )
#     dev_sign = int(info_dev["chosen_sign"])


#     # --- prep external ---
#     ext_df = ext_df.reset_index(drop=True).copy()
#     needed = ["Disease_Duration", event_col] + list(vars_final)
#     if weight_col is not None and weight_col in ext_df.columns:
#         needed = needed + [weight_col]

#     missing = [c for c in needed if c not in ext_df.columns]
#     if missing:
#         raise ValueError(f"External df missing columns: {missing}")

#     rng = np.random.default_rng(random_state)
#     n = len(ext_df)

#     reps = {
#         "cindex": [],
#         "slope": [],
#         "citl_frozen": [],
#         "citl_updated": [],
#         "mean_pred_frozen": [],
#         "mean_pred_updated": [],
#         "obs_risk_km": [],
#         "baseline_factor": [],
#     }

#     for _ in range(n_boot):
#         idx = rng.integers(0, n, size=n)
#         boot = ext_df.iloc[idx].reset_index(drop=True)

#         r_boot = to_r_df(boot[needed])

       
#         score = risk_score_stpm2(
#             model_final, r_boot,
#             vars_final=vars_final,
#             event_col=event_col,
#             sign=dev_sign,
#             return_details=False
#         )

#         w_boot = np.asarray(r_boot.rx2(weight_col), dtype=float) if (weight_col is not None and weight_col in boot.columns) else None


#         score = np.asarray(score, float) * float(shrinkage)

#         reps["cindex"].append(cindex_from_risk_score(r_boot, score))
#         reps["slope"].append(slope_from_risk_score(r_boot, score))

#         # predicted survival at t0
#         S_pred = predict_surv_at_t0(model_final, r_boot, t0_months)
#         risk_frozen = 1.0 - S_pred
#         mean_pred_frozen = weighted_mean_np(risk_frozen, w_boot)

#         # observed KM at t0
#         S_obs = km_surv_at_t0(r_boot, t0_months)
#         if not np.isfinite(S_obs):
#             # skip (rare): no KM estimate at exactly t0 within this bootstrap resample
#             continue

#         p_obs = 1.0 - float(S_obs)
#         reps["obs_risk_km"].append(p_obs)
#         reps["mean_pred_frozen"].append(mean_pred_frozen)
#         reps["citl_frozen"].append(citl(p_obs, mean_pred_frozen))

#         # baseline update factor and updated risks
#         S_obs_c = float(np.clip(S_obs, 1e-12, 1 - 1e-12))
#         S_bar_c = float(np.clip(np.mean(S_pred), 1e-12, 1 - 1e-12))
#         alpha = np.log(-np.log(S_obs_c)) - np.log(-np.log(S_bar_c))
#         factor = float(np.exp(alpha))
#         reps["baseline_factor"].append(factor)

#         if baseline_update:
#             S_upd = np.power(S_pred, factor)
#             risk_updated = 1.0 - S_upd
#         else:
#             risk_updated = risk_frozen

#         mean_pred_updated = weighted_mean_np(risk_updated, w_boot)
#         reps["mean_pred_updated"].append(mean_pred_updated)
#         reps["citl_updated"].append(citl(p_obs, mean_pred_updated))

#     # convert to arrays
#     reps_np = {k: np.asarray(v, float) for k, v in reps.items()}

#     def pct_ci(x):
#         x = x[np.isfinite(x)]
#         if len(x) == 0:
#             return (np.nan, np.nan)
#         return tuple(np.percentile(x, [2.5, 97.5]).astype(float))

#     cis = {k: pct_ci(v) for k, v in reps_np.items()}

#     out = {
#         "dev_score_sign": dev_sign,
#         "n_boot_requested": int(n_boot),
#         "n_boot_used_for_km_based": int(len(reps_np["obs_risk_km"])),  # may be < n_boot if KM missing at t0 sometimes
#         "ci": cis,
#         "point_estimate_from_full_external": None,  # optional: fill by calling external_validate_stpm2_corrected once
#     }

#     if return_replicates:
#         out["replicates"] = reps_np

#     return out


# # ------------------------------------------------------------
# # Optional: pretty summary table (CI separate column)
# # ------------------------------------------------------------
# def make_external_ci_summary_table(res_ext: dict, boot_cis: dict) -> pd.DataFrame:
#     ci = boot_cis["ci"]

#     def fmt(x): 
#         return "" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{float(x):.3f}"

#     def fmt_ci(name):
#         lo, hi = ci.get(name, (np.nan, np.nan))
#         if not np.isfinite(lo) or not np.isfinite(hi):
#             return ""
#         return f"[{lo:.3f}, {hi:.3f}]"

#     rows = [
#         ("C-index (external)", res_ext.get("cindex_external"), fmt_ci("cindex")),
#         ("Calibration slope (external)", res_ext.get("slope_external"), fmt_ci("slope")),
#         (f"CITL at {res_ext.get('t0_months')}m (frozen)", res_ext.get("citl_external_frozen"), fmt_ci("citl_frozen")),
#         (f"CITL at {res_ext.get('t0_months')}m (baseline-updated)", res_ext.get("citl_external_updated"), fmt_ci("citl_updated")),
#         (f"Observed risk at {res_ext.get('t0_months')}m (KM)", res_ext.get("obs_risk_km"), fmt_ci("obs_risk_km")),
#         (f"Mean predicted risk at {res_ext.get('t0_months')}m (frozen)", res_ext.get("mean_pred_risk_frozen"), fmt_ci("mean_pred_frozen")),
#         (f"Mean predicted risk at {res_ext.get('t0_months')}m (baseline-updated)", res_ext.get("mean_pred_risk_updated"), fmt_ci("mean_pred_updated")),
#         ("Baseline update factor", res_ext.get("baseline_factor"), fmt_ci("baseline_factor")),
#     ]

#     df = pd.DataFrame(rows, columns=["Metric", "Value", "95% CI"])
#     df["Value"] = df["Value"].map(fmt)
#     return df



# # ----------------------------
# # Data prep (drop IDs; enforce numeric)
# # ----------------------------
# def prep_df(df: pd.DataFrame, covariates: list[str], weight_col: str | None = None) -> pd.DataFrame:
#     keep = ["Disease_Duration", "Event"] + covariates
#     if weight_col is not None and weight_col in df.columns:
#         keep = keep + [weight_col]

#     out = df.copy()
#     out = out.loc[:, [c for c in keep if c in out.columns]].copy()
#     out["Event"] = out["Event"].astype(int)

#     if weight_col is not None and weight_col in out.columns:
#         out[weight_col] = pd.to_numeric(out[weight_col], errors="coerce").fillna(0.0)

#     return out


# def _to_r_df(df: pd.DataFrame):
#     with localconverter(default_converter + pandas2ri.converter):
#         return pandas2ri.py2rpy(df)


# def fit_stpm2_and_aic(
#     r_df, 
#     rhs_vars: list[str], 
#     spline_df: int,
#     weight_col: str | None = "weight",
#     robust: bool = True
#     ) -> tuple[object, float, str]:
#         rhs = " + ".join(rhs_vars) if rhs_vars else "1"
#         formula_str = f"Surv(Disease_Duration, Event==1) ~ {rhs}"

#         if weight_col is not None:
#             model = rstpm2.stpm2(
#                 Formula(formula_str),
#                 data=r_df,
#                 df=spline_df,
#                 weights=r_df.rx2(weight_col),
#                 robust=robust
#             )
#         else:
#             model = rstpm2.stpm2(Formula(formula_str), data=r_df, df=spline_df)
        
#         aic = float(stats.AIC(model)[0])
#         return model, aic, formula_str


# def citl_from_obs_and_pred(p_obs, p_pred_mean):
#     p_obs = float(np.clip(p_obs, 1e-12, 1-1e-12))
#     p_pred_mean = float(np.clip(p_pred_mean, 1e-12, 1-1e-12))
#     return float(np.log(p_obs/(1-p_obs)) - np.log(p_pred_mean/(1-p_pred_mean)))


# def make_external_summary_table(
#     res_ext: dict,
#     t0_months: int,
#     boot_cis: dict | None = None,
#     digits: int = 3,
# ) -> pd.DataFrame:
#     """
#     Build a slide/manuscript-ready external validation summary table.

#     If boot_cis is provided (from external_bootstrap_cis), 95% CIs are filled.
#     Otherwise, the CI column is left blank.
#     """

#     # ---- helpers ----
#     def fmt_val(x, d=digits):
#         if isinstance(x, (float, int, np.floating)) and np.isfinite(x):
#             return f"{float(x):.{d}f}"
#         return "" if x is None else x

#     def fmt_ci(name, d=digits):
#         if boot_cis is None:
#             return ""
#         lo, hi = boot_cis.get("ci", {}).get(name, (np.nan, np.nan))
#         if not np.isfinite(lo) or not np.isfinite(hi):
#             return ""
#         return f"[{lo:.{d}f}, {hi:.{d}f}]"

#     rows = []

#     def add(metric, value=None, ci_name=None, d=digits):
#         rows.append({
#             "Metric": metric,
#             "Value": fmt_val(value, d),
#             "95% CI": fmt_ci(ci_name, d) if ci_name else ""
#         })

#     # ---- model / setup ----
#     add("External time horizon (months)", t0_months, d=0)
#     add("Model variables", ", ".join(res_ext.get("vars_final", [])))
#     add("Shrinkage factor (uniform)", res_ext.get("shrinkage", np.nan))

#     # ---- discrimination ----
#     add("C-index (external)", res_ext.get("cindex_external"), "cindex")
#     add("Calibration slope (external)", res_ext.get("slope_external"), "slope")

#     # ---- observed risk ----
#     add(f"Observed KM risk at {t0_months} months",
#         res_ext.get("obs_risk_km"), "obs_risk_km")

#     # ---- absolute calibration (frozen) ----
#     add(f"Mean predicted risk at {t0_months} months (frozen)",
#         res_ext.get("mean_pred_risk_frozen"), "mean_pred_frozen")
#     add(f"CITL at {t0_months} months (frozen)",
#         res_ext.get("citl_external_frozen"), "citl_frozen")

#     # ---- baseline updating ----
#     add("Baseline update factor",
#         res_ext.get("baseline_factor"), "baseline_factor")
#     add(f"Mean predicted risk at {t0_months} months (baseline-updated)",
#         res_ext.get("mean_pred_risk_updated"), "mean_pred_updated")
#     add(f"CITL at {t0_months} months (baseline-updated)",
#         res_ext.get("citl_external_updated"), "citl_updated")

#     return pd.DataFrame(rows, columns=["Metric", "Value", "95% CI"])


# @dataclass
# class SelectionResult:
#     vars_final: list[str]
#     spline_df: int
#     aic: float
#     formula: str

##### corrected?

In [ ]:
# ============================================================
# WEIGHT-AWARE STPM2 VALIDATION / BOOTSTRAP UTILITIES
# CLEANED, MERGED, END-TO-END VERSION
# ------------------------------------------------------------
# Key features
#   - weighted stpm2 fitting
#   - safe complete-case handling before every fit
#   - weighted C-index / slope / KM / CITL
#   - weighted score-sign choice
#   - weighted bootstrap internal validation
#   - weighted external validation
#
# Important design choice
#   - If you want UNWEIGHTED development but WEIGHTED external validation:
#       * call internal/bootstrap functions with weight_col=None
#       * call external validation with weight_col="weight"
#
# Assumes already imported:
#   import numpy as np
#   import pandas as pd
#   import hashlib
#   from dataclasses import dataclass
#   from collections import Counter
#   import matplotlib.pyplot as plt
#   from matplotlib.ticker import FuncFormatter
#
#   import rpy2.robjects as ro
#   from rpy2.robjects import Formula, default_converter, pandas2ri
#   from rpy2.robjects.vectors import FloatVector, IntVector
#   from rpy2.robjects.conversion import localconverter
#
#   from rpy2.robjects.packages import importr
#   rstpm2 = importr("rstpm2")
#   survival = importr("survival")
#   stats = importr("stats")
# ============================================================

R = ro.r

# ============================================================
# R snippets
# ============================================================
_r_conc_extract = R("""function(conc_obj) as.numeric(conc_obj$concordance[1])""")

_r_predict_surv_t0 = R("""
function(model, newdata, t0){
  n <- nrow(newdata)
  tmp <- transform(newdata, t0=rep(as.numeric(t0), n))
  as.numeric(predict(model, newdata=tmp, type="surv", timevar="t0"))
}
""")

_r_km_surv_t0_w = R("""
function(df, t0, weight_col="weight"){
  w <- as.numeric(df[[weight_col]])
  fit <- survival::survfit(
    survival::Surv(Disease_Duration, Event==1) ~ 1,
    data=df,
    weights=w
  )
  ss <- summary(fit, times=as.numeric(t0))
  if(length(ss$surv)==0) return(NA_real_)
  as.numeric(ss$surv[1])
}
""")

_r_km_surv_t0_unw = R("""
function(df, t0){
  fit <- survival::survfit(
    survival::Surv(Disease_Duration, Event==1) ~ 1,
    data=df
  )
  ss <- summary(fit, times=as.numeric(t0))
  if(length(ss$surv)==0) return(NA_real_)
  as.numeric(ss$surv[1])
}
""")

_r_km_risk_ci_t0_robust_w = R("""
function(df, t0, weight_col="weight"){
  w <- as.numeric(df[[weight_col]])
  fit <- survival::survfit(
    survival::Surv(Disease_Duration, Event==1) ~ 1,
    data=df,
    weights=w
  )

  tt <- fit$time
  if(length(tt)==0) return(c(NA_real_, NA_real_, NA_real_))
  t_use <- max(tt[tt <= as.numeric(t0)], na.rm=TRUE)
  if(!is.finite(t_use)) return(c(NA_real_, NA_real_, NA_real_))

  ss <- summary(fit, times=t_use)
  if(length(ss$surv)==0) return(c(NA_real_, NA_real_, NA_real_))

  S  <- as.numeric(ss$surv[1])
  Lo <- as.numeric(ss$lower[1])
  Hi <- as.numeric(ss$upper[1])

  risk    <- 1 - S
  risk_lo <- 1 - Hi
  risk_hi <- 1 - Lo
  c(risk, risk_lo, risk_hi)
}
""")

_r_km_risk_ci_t0_robust_unw = R("""
function(df, t0){
  fit <- survival::survfit(
    survival::Surv(Disease_Duration, Event==1) ~ 1,
    data=df
  )

  tt <- fit$time
  if(length(tt)==0) return(c(NA_real_, NA_real_, NA_real_))
  t_use <- max(tt[tt <= as.numeric(t0)], na.rm=TRUE)
  if(!is.finite(t_use)) return(c(NA_real_, NA_real_, NA_real_))

  ss <- summary(fit, times=t_use)
  if(length(ss$surv)==0) return(c(NA_real_, NA_real_, NA_real_))

  S  <- as.numeric(ss$surv[1])
  Lo <- as.numeric(ss$lower[1])
  Hi <- as.numeric(ss$upper[1])

  risk    <- 1 - S
  risk_lo <- 1 - Hi
  risk_hi <- 1 - Lo
  c(risk, risk_lo, risk_hi)
}
""")

_r_ipcw_recal_t0 = R("""
function(time, status, p_pred, t0, estimate_slope=FALSE){
  time   <- as.numeric(time)
  status <- as.integer(status)
  status <- ifelse(status == 1L, 1L, 0L)
  status <- as.integer(status)

  p_pred <- as.numeric(p_pred)
  t0     <- as.numeric(t0)

  eps <- 1e-12
  p_pred <- pmin(pmax(p_pred, eps), 1 - eps)

  censor_event <- 1L - status
  fitG <- survival::survfit(survival::Surv(time, censor_event) ~ 1)

  G_at <- function(u){
    ss <- summary(fitG, times=u)
    if(length(ss$surv)==0){
      return(tail(fitG$surv, 1))
    }
    return(ss$surv[1])
  }

  w <- rep(0, length(time))
  Y <- ifelse(time <= t0 & status == 1L, 1L, 0L)
  Y <- as.integer(Y)

  idx_event <- which(time <= t0 & status == 1L)
  if(length(idx_event) > 0){
    Gi <- sapply(time[idx_event], G_at)
    w[idx_event] <- 1 / pmax(Gi, eps)
  }

  idx_free <- which(time >= t0)
  if(length(idx_free) > 0){
    Gt0 <- G_at(t0)
    w[idx_free] <- 1 / pmax(Gt0, eps)
  }

  lp_pred <- qlogis(p_pred)

  if(!estimate_slope){
    fit <- stats::glm(Y ~ 1 + offset(lp_pred),
                      family=stats::quasibinomial(),
                      weights=w)
    alpha <- stats::coef(fit)[1]
    return(list(alpha=as.numeric(alpha), model=fit))
  } else {
    fit <- stats::glm(Y ~ lp_pred,
                      family=stats::quasibinomial(),
                      weights=w)
    alpha <- stats::coef(fit)[1]
    beta  <- stats::coef(fit)[2]
    return(list(alpha=as.numeric(alpha), beta=as.numeric(beta), model=fit))
  }
}
""")

_r_ipcw_recal_t0_w = R("""
function(time, status, p_pred, t0, base_w=NULL, estimate_slope=FALSE){
  time   <- as.numeric(time)
  status <- as.integer(status)
  status <- ifelse(status == 1L, 1L, 0L)
  status <- as.integer(status)

  p_pred <- as.numeric(p_pred)
  t0     <- as.numeric(t0)

  if(is.null(base_w)){
    base_w <- rep(1, length(time))
  } else {
    base_w <- as.numeric(base_w)
  }

  eps <- 1e-12
  p_pred <- pmin(pmax(p_pred, eps), 1 - eps)

  censor_event <- 1L - status
  fitG <- survival::survfit(survival::Surv(time, censor_event) ~ 1)

  G_at <- function(u){
    ss <- summary(fitG, times=u)
    if(length(ss$surv)==0){
      return(tail(fitG$surv, 1))
    }
    return(ss$surv[1])
  }

  Y <- ifelse(time <= t0 & status == 1L, 1L, 0L)
  Y <- as.integer(Y)

  w_ipcw <- rep(0, length(time))

  idx_event <- which(time <= t0 & status == 1L)
  if(length(idx_event) > 0){
    Gi <- sapply(time[idx_event], G_at)
    w_ipcw[idx_event] <- 1 / pmax(Gi, eps)
  }

  idx_free <- which(time >= t0)
  if(length(idx_free) > 0){
    Gt0 <- G_at(t0)
    w_ipcw[idx_free] <- 1 / pmax(Gt0, eps)
  }

  w <- base_w * w_ipcw
  lp_pred <- qlogis(p_pred)

  if(!estimate_slope){
    fit <- stats::glm(Y ~ 1 + offset(lp_pred),
                      family=stats::quasibinomial(),
                      weights=w)
    alpha <- stats::coef(fit)[1]
    return(list(alpha=as.numeric(alpha), model=fit))
  } else {
    fit <- stats::glm(Y ~ lp_pred,
                      family=stats::quasibinomial(),
                      weights=w)
    alpha <- stats::coef(fit)[1]
    beta  <- stats::coef(fit)[2]
    return(list(alpha=as.numeric(alpha), beta=as.numeric(beta), model=fit))
  }
}
""")

# ============================================================
# Data conversion / cleaning helpers
# ============================================================
def to_r_df(df: pd.DataFrame):
    with localconverter(default_converter + pandas2ri.converter):
        return pandas2ri.py2rpy(df)

def _to_r_df(df: pd.DataFrame):
    with localconverter(default_converter + pandas2ri.converter):
        return pandas2ri.py2rpy(df)

def _parse_rhs_vars(formula_str: str) -> list[str]:
    rhs = formula_str.split("~", 1)[1].strip()
    if rhs == "1":
        return []
    return [x.strip() for x in rhs.split("+")]

def build_analysis_df(
    df_python: pd.DataFrame,
    rhs_vars: list[str],
    weight_col: str | None = "weight",
    time_col: str = "Disease_Duration",
    event_col: str = "Event",
) -> pd.DataFrame:
    needed = [time_col, event_col] + list(rhs_vars)
    if weight_col is not None and weight_col in df_python.columns:
        needed = needed + [weight_col]

    dat = df_python.loc[:, needed].copy()

    # coerce to numeric
    for c in [time_col, event_col] + list(rhs_vars):
        dat[c] = pd.to_numeric(dat[c], errors="coerce")

    if weight_col is not None and weight_col in dat.columns:
        dat[weight_col] = pd.to_numeric(dat[weight_col], errors="coerce")

    dat = dat.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    # keep positive finite weights only
    if weight_col is not None and weight_col in dat.columns:
        dat = dat[np.isfinite(dat[weight_col]) & (dat[weight_col] > 0)].reset_index(drop=True)

    dat[event_col] = dat[event_col].astype(int)
    return dat

def fit_stpm2_on_dataframe(
    df_python: pd.DataFrame,
    formula_str: str,
    spline_df: int,
    weight_col: str | None = "weight",
    robust: bool = True,
    time_col: str = "Disease_Duration",
    event_col: str = "Event",
):
    rhs_vars = _parse_rhs_vars(formula_str)
    dat = build_analysis_df(
        df_python=df_python,
        rhs_vars=rhs_vars,
        weight_col=weight_col,
        time_col=time_col,
        event_col=event_col,
    ).reset_index(drop=True)

    r_df = to_r_df(dat)

    if weight_col is None or weight_col not in dat.columns:
        model = rstpm2.stpm2(
            Formula(formula_str),
            data=r_df,
            df=spline_df
        )
    else:
        model = rstpm2.stpm2(
            Formula(formula_str),
            data=r_df,
            df=spline_df,
            weights=r_df.rx2(weight_col),
            robust=robust
        )
    return model, dat, r_df

def fit_stpm2_and_aic(
    df_python: pd.DataFrame,
    rhs_vars: list[str],
    spline_df: int,
    weight_col: str | None = "weight",
    robust: bool = True,
    time_col: str = "Disease_Duration",
    event_col: str = "Event",
) -> tuple[object, float, str]:
    rhs = " + ".join(rhs_vars) if rhs_vars else "1"
    formula_str = f"Surv({time_col}, {event_col}==1) ~ {rhs}"
    model, dat, r_df = fit_stpm2_on_dataframe(
        df_python=df_python,
        formula_str=formula_str,
        spline_df=spline_df,
        weight_col=weight_col,
        robust=robust,
        time_col=time_col,
        event_col=event_col,
    )
    aic = float(stats.AIC(model)[0])
    return model, aic, formula_str


# ============================================================
# Basic utilities
# ============================================================
def weighted_mean_np(x: np.ndarray, w: np.ndarray | None = None) -> float:
    x = np.asarray(x, float)
    if w is None:
        return float(np.mean(x))
    w = np.asarray(w, float)
    ok = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not np.any(ok):
        return np.nan
    return float(np.sum(w[ok] * x[ok]) / np.sum(w[ok]))

def _logit(p: float) -> float:
    p = float(np.clip(p, 1e-12, 1 - 1e-12))
    return float(np.log(p / (1 - p)))

def ci_str(lo, hi, digits=3):
    if lo is None or hi is None or (not np.isfinite(lo)) or (not np.isfinite(hi)):
        return ""
    return f"[{lo:.{digits}f}, {hi:.{digits}f}]"

def _get_weights_from_r(r_df, weight_col: str | None = "weight") -> np.ndarray | None:
    if weight_col is None:
        return None
    try:
        return np.asarray(r_df.rx2(weight_col), dtype=float)
    except Exception:
        return None

# ============================================================
# KM helpers
# ============================================================
def km_surv_at_t0(r_df, t0_months: float, weight_col: str | None = "weight") -> float:
    if weight_col is None:
        return float(_r_km_surv_t0_unw(r_df, float(t0_months))[0])
    return float(_r_km_surv_t0_w(r_df, float(t0_months), weight_col)[0])

def km_risk_ci_at_t0(
    r_df_sub,
    t0_months: int,
    weight_col: str | None = "weight"
) -> tuple[float, float, float]:
    if weight_col is None:
        out = np.asarray(_r_km_risk_ci_t0_robust_unw(r_df_sub, float(t0_months)), dtype=float)
    else:
        out = np.asarray(_r_km_risk_ci_t0_robust_w(r_df_sub, float(t0_months), weight_col), dtype=float)

    if np.any(~np.isfinite(out)):
        return (np.nan, np.nan, np.nan)
    return (float(out[0]), float(out[1]), float(out[2]))

# ============================================================
# Prediction helpers
# ============================================================
def predict_surv_at_t0(model, r_df, t0_months: float) -> np.ndarray:
    return np.asarray(_r_predict_surv_t0(model, r_df, float(t0_months)), dtype=float)

def predict_pi_from_stpm2(model, r_df, vars_final: list[str]) -> np.ndarray:
    coef_vec = ro.r["coef"](model)
    coef_names = list(map(str, ro.r["names"](coef_vec)))
    b = np.asarray(list(coef_vec), dtype=float)
    coef_map = dict(zip(coef_names, b))

    missing = [v for v in vars_final if v not in coef_map]
    if missing:
        raise ValueError(
            f"vars_final not found in coef(model): {missing}\n"
            f"First 40 coef names: {coef_names[:40]}"
        )

    Xcols = []
    for v in vars_final:
        Xcols.append(np.asarray(r_df.rx2(v), dtype=float))
    X = np.column_stack(Xcols)

    beta = np.asarray([coef_map[v] for v in vars_final], dtype=float)
    return X @ beta

def _event_mask_from_r(r_df, event_col: str = "Event") -> np.ndarray:
    ev = np.asarray(r_df.rx2(event_col), dtype=float)
    return (ev == 1)

# ============================================================
# Score sign helpers
# ============================================================
def choose_score_sign_from_lp(
    lp: np.ndarray,
    event_mask: np.ndarray,
    weights: np.ndarray | None = None,
) -> int:
    lp = np.asarray(lp, float)
    event_mask = np.asarray(event_mask, bool)

    if weights is None:
        m1 = float(np.mean(lp[event_mask])) if np.any(event_mask) else np.nan
        m0 = float(np.mean(lp[~event_mask])) if np.any(~event_mask) else np.nan
    else:
        w = np.asarray(weights, float)
        m1 = weighted_mean_np(lp[event_mask], w[event_mask]) if np.any(event_mask) else np.nan
        m0 = weighted_mean_np(lp[~event_mask], w[~event_mask]) if np.any(~event_mask) else np.nan

    if not np.isfinite(m1) or not np.isfinite(m0):
        return +1
    return +1 if (m1 > m0) else -1

def risk_score_stpm2(
    model,
    r_df,
    vars_final: list[str],
    event_col: str = "Event",
    weight_col: str | None = "weight",
    sign: int | None = None,
    return_details: bool = False,
):
    pi = predict_pi_from_stpm2(model, r_df, vars_final=vars_final)
    event_mask = _event_mask_from_r(r_df, event_col=event_col)
    w = _get_weights_from_r(r_df, weight_col=weight_col)

    if sign is None:
        sign = choose_score_sign_from_lp(pi, event_mask, weights=w)

    score = sign * pi

    if not return_details:
        return score

    if w is None:
        m1_pi = float(np.mean(pi[event_mask])) if np.any(event_mask) else np.nan
        m0_pi = float(np.mean(pi[~event_mask])) if np.any(~event_mask) else np.nan
        m1_sc = float(np.mean(score[event_mask])) if np.any(event_mask) else np.nan
        m0_sc = float(np.mean(score[~event_mask])) if np.any(~event_mask) else np.nan
    else:
        m1_pi = weighted_mean_np(pi[event_mask], w[event_mask]) if np.any(event_mask) else np.nan
        m0_pi = weighted_mean_np(pi[~event_mask], w[~event_mask]) if np.any(~event_mask) else np.nan
        m1_sc = weighted_mean_np(score[event_mask], w[event_mask]) if np.any(event_mask) else np.nan
        m0_sc = weighted_mean_np(score[~event_mask], w[~event_mask]) if np.any(~event_mask) else np.nan

    return score, {
        "chosen_sign": int(sign),
        "mean_pi_event1": m1_pi,
        "mean_pi_event0": m0_pi,
        "mean_score_event1": m1_sc,
        "mean_score_event0": m0_sc,
    }

# ============================================================
# Performance metrics
# ============================================================
def cindex_from_risk_score(
    r_df,
    score: np.ndarray,
    weight_col: str | None = "weight",
) -> float:
    r_tmp = R("transform")(r_df, score=FloatVector(np.asarray(score, float).tolist()))

    if weight_col is None:
        conc_obj = survival.concordance(
            Formula("Surv(Disease_Duration, Event==1) ~ I(-score)"),
            data=r_tmp
        )
    else:
        conc_obj = survival.concordance(
            Formula("Surv(Disease_Duration, Event==1) ~ I(-score)"),
            data=r_tmp,
            weights=r_tmp.rx2(weight_col)
        )

    return float(_r_conc_extract(conc_obj)[0])

def slope_from_risk_score(
    r_df,
    score: np.ndarray,
    weight_col: str | None = "weight",
) -> float:
    r_tmp = R("transform")(r_df, score=FloatVector(np.asarray(score, float).tolist()))

    if weight_col is None:
        cal = survival.coxph(
            Formula("Surv(Disease_Duration, Event==1) ~ score"),
            data=r_tmp
        )
    else:
        cal = survival.coxph(
            Formula("Surv(Disease_Duration, Event==1) ~ score"),
            data=r_tmp,
            weights=r_tmp.rx2(weight_col),
            robust=True
        )

    return float(np.asarray(R["coef"](cal), dtype=float)[0])

# ============================================================
# IPCW calibration helpers
# ============================================================
def ipcw_calibration_intercept_t0_stpm2(
    model,
    r_df,
    t0_months: float,
    event_col: str = "Event",
    time_col: str = "Disease_Duration",
) -> float:
    S_pred = predict_surv_at_t0(model, r_df, t0_months)
    p_pred = 1.0 - np.asarray(S_pred, float)
    p_pred = np.clip(p_pred, 1e-12, 1 - 1e-12)

    time_np   = np.asarray(r_df.rx2(time_col), dtype=float)
    status_np = np.asarray(r_df.rx2(event_col), dtype=int)

    time_r   = FloatVector(time_np.tolist())
    status_r = IntVector(status_np.tolist())
    p_pred_r = FloatVector(p_pred.tolist())

    out = _r_ipcw_recal_t0(time_r, status_r, p_pred_r, float(t0_months), False)
    return float(out.rx2("alpha")[0])

def ipcw_intercept_and_slope_t0_stpm2(
    model,
    r_df,
    t0_months: float,
    event_col: str = "Event",
    time_col: str = "Disease_Duration",
):
    S_pred = predict_surv_at_t0(model, r_df, t0_months)
    p_pred = 1.0 - np.asarray(S_pred, float)
    p_pred = np.clip(p_pred, 1e-12, 1 - 1e-12)

    time_np   = np.asarray(r_df.rx2(time_col), dtype=float)
    status_np = np.asarray(r_df.rx2(event_col), dtype=int)

    time_r   = FloatVector(time_np.tolist())
    status_r = IntVector(status_np.tolist())
    p_pred_r = FloatVector(p_pred.tolist())

    out = _r_ipcw_recal_t0(time_r, status_r, p_pred_r, float(t0_months), True)
    alpha = float(out.rx2("alpha")[0])
    beta  = float(out.rx2("beta")[0])
    return alpha, beta

def ipcw_citl_from_pred_risk(
    r_df,
    p_pred: np.ndarray,
    t0_months: float,
    event_col: str = "Event",
    time_col: str = "Disease_Duration",
    weight_col: str | None = "weight",
) -> float:
    p_pred = np.asarray(p_pred, float)
    p_pred = np.clip(p_pred, 1e-12, 1 - 1e-12)

    time_np   = np.asarray(r_df.rx2(time_col), dtype=float)
    status_np = np.asarray(r_df.rx2(event_col), dtype=int)

    time_r   = FloatVector(time_np.tolist())
    status_r = IntVector(status_np.tolist())
    p_pred_r = FloatVector(p_pred.tolist())

    if weight_col is not None:
        base_w_np = np.asarray(r_df.rx2(weight_col), dtype=float)
        base_w_r = FloatVector(base_w_np.tolist())
    else:
        base_w_r = ro.NULL

    out = _r_ipcw_recal_t0_w(time_r, status_r, p_pred_r, float(t0_months), base_w_r, False)
    return float(out.rx2("alpha")[0])

# ============================================================
# CITL helpers
# ============================================================
def citl_at_t0(
    model,
    r_df,
    t0_months: int,
    weight_col: str | None = "weight",
) -> float:
    S_pred = predict_surv_at_t0(model, r_df, t0_months)
    p_pred = 1.0 - S_pred
    w = _get_weights_from_r(r_df, weight_col=weight_col)

    p_bar = weighted_mean_np(p_pred, w)
    S_km = km_surv_at_t0(r_df, t0_months, weight_col=weight_col)
    if not np.isfinite(S_km):
        return np.nan

    p_obs = 1.0 - float(S_km)
    return float(_logit(p_obs) - _logit(p_bar))

def delta_logit_km_vs_meanpred_at_t0(
    model,
    r_df,
    t0_months: int,
    weight_col: str | None = "weight"
) -> float:
    return citl_at_t0(model, r_df, t0_months, weight_col=weight_col)

def citl_ipcw_at_t0(
    model,
    r_df,
    t0_months: int,
) -> float:
    return ipcw_calibration_intercept_t0_stpm2(model, r_df, t0_months)

# ============================================================
# Shrinkage
# ============================================================
def uniform_shrinkage_from_slope(slope_corr: float) -> float:
    if not np.isfinite(slope_corr) or slope_corr <= 0:
        return 1.0
    return float(min(1.0, slope_corr))

# ============================================================
# Apparent evaluation
# ============================================================
def evaluate_stpm2_apparent(
    model,
    r_df,
    vars_final,
    t0_months: int = 24,
    event_col: str = "Event",
    weight_col: str | None = "weight",
    sign: int | None = None,
    verbose: bool = True,
):
    score, info = risk_score_stpm2(
        model, r_df,
        vars_final=vars_final,
        event_col=event_col,
        weight_col=weight_col,
        sign=sign,
        return_details=True
    )

    c = cindex_from_risk_score(r_df, score, weight_col=weight_col)
    s = slope_from_risk_score(r_df, score, weight_col=weight_col)
    citl_ipcw = ipcw_citl_from_pred_risk(
        r_df,
        p_pred=1.0 - predict_surv_at_t0(model, r_df, t0_months),
        t0_months=t0_months,
        event_col=event_col,
        weight_col=weight_col,
    )
    citl_simple = citl_at_t0(model, r_df, t0_months, weight_col=weight_col)

    if verbose:
        print("Score sign chosen:", info["chosen_sign"])
        print("Mean PI event=1 / event=0:", info["mean_pi_event1"], info["mean_pi_event0"])
        print("Mean score event=1 / event=0:", info["mean_score_event1"], info["mean_score_event0"])
        print("C-index:", c)
        print("Slope:", s)
        print("CITL IPCW:", citl_ipcw)
        print("CITL simple:", citl_simple)

    return {
        "cindex": c,
        "slope": s,
        "citl_ipcw": citl_ipcw,
        "citl_simple": citl_simple,
        "score_sign": info["chosen_sign"],
        "details": info,
    }

# ============================================================
# Calibration bins
# ============================================================
def calibration_bins_km_robust(
    r_df,
    pred_risk: np.ndarray,
    t0_months: int,
    n_bins: int = 5,
    weight_col: str | None = "weight",
) -> pd.DataFrame:
    pr = np.asarray(pred_risk, float)
    df = pd.DataFrame({"pred": pr})
    df = df[np.isfinite(df["pred"])].copy()
    if df.empty:
        return pd.DataFrame()

    try:
        df["bin"] = pd.qcut(df["pred"], q=n_bins, duplicates="drop")
        if df["bin"].isna().all() or len(df["bin"].cat.categories) < 2:
            raise ValueError("qcut collapsed bins")
    except Exception:
        r = df["pred"].rank(method="average")
        df["bin"] = pd.qcut(r, q=min(n_bins, df.shape[0]), duplicates="drop")

    rows = []
    for b in df["bin"].cat.categories:
        mask = (df["bin"] == b).to_numpy()
        idx0 = np.where(mask)[0]
        if len(idx0) == 0:
            continue

        r_sub = r_df.rx(IntVector((idx0 + 1).tolist()), True)
        risk, lo, hi = km_risk_ci_at_t0(r_sub, t0_months, weight_col=weight_col)
        if not np.isfinite(risk):
            continue

        if weight_col is not None:
            w_sub = np.asarray(r_sub.rx2(weight_col), dtype=float)
            pred_sub = df.loc[mask, "pred"].to_numpy(float)
            mean_pred = weighted_mean_np(pred_sub, w_sub)
        else:
            mean_pred = float(df.loc[mask, "pred"].mean())

        rows.append({
            "bin": str(b),
            "n": int(mask.sum()),
            "mean_pred": float(mean_pred),
            "obs_km": float(risk),
            "obs_lo": float(lo),
            "obs_hi": float(hi),
        })

    return pd.DataFrame(rows)

# ============================================================
# Data prep
# ============================================================
def prep_df(
    df: pd.DataFrame,
    covariates: list[str],
    weight_col: str | None = "weight",
    time_col: str = "Disease_Duration",
    event_col: str = "Event",
) -> pd.DataFrame:
    keep = [time_col, event_col] + covariates
    if weight_col is not None and weight_col in df.columns:
        keep = keep + [weight_col]

    out = df.copy()
    out = out.loc[:, [c for c in keep if c in out.columns]].copy()
    out[time_col] = pd.to_numeric(out[time_col], errors="coerce")
    out[event_col] = pd.to_numeric(out[event_col], errors="coerce").astype("Int64")

    for c in covariates:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    if weight_col is not None and weight_col in out.columns:
        out[weight_col] = pd.to_numeric(out[weight_col], errors="coerce")

    return out

# ============================================================
# Selection / result classes
# ============================================================
@dataclass
class SelectionResult:
    vars_final: list[str]
    spline_df: int
    aic: float
    formula: str

@dataclass
class FullStrategyValidationResult:
    best_vars_full: list[str]
    best_spline_df_full: int
    best_aic_full: float
    best_formula_full: str

    cindex_app: float
    slope_app: float
    citl_app: float

    cindex_corr: float
    slope_corr: float
    citl_corr: float

    cindex_ci: tuple[float, float]
    slope_ci: tuple[float, float]
    citl_ci: tuple[float, float]

    n_boot_success: int
    n_boot_fail: int

def _model_signature(vars_list: list[str], spline_df: int) -> str:
    key = f"df={spline_df}|" + "+".join(vars_list)
    return hashlib.md5(key.encode("utf-8")).hexdigest()

# ============================================================
# Backward AIC selection
# ============================================================
def backward_elimination_aic(
    df_python: pd.DataFrame,
    covariates: list[str],
    df_candidates: list[int],
    hard_include: list[str],
    hard_exclude: list[str] | None = None,
    weight_col: str | None = "weight",
    robust: bool = True,
    time_col: str = "Disease_Duration",
    event_col: str = "Event",
) -> SelectionResult:
    hard_exclude = list(hard_exclude or [])

    start_vars = [v for v in covariates if v not in set(hard_exclude)]
    for v in hard_include:
        if v not in start_vars and v not in set(hard_exclude):
            start_vars.append(v)

    forced = set(hard_include)
    current_vars = start_vars[:]
    best: SelectionResult | None = None

    def better(a: SelectionResult, b: SelectionResult | None) -> bool:
        if b is None:
            return True
        return a.aic < b.aic - 1e-12

    while len(current_vars) >= 1:
        candidates: list[SelectionResult] = []

        for ddf in df_candidates:
            _, aic, fml = fit_stpm2_and_aic(
                df_python=df_python,
                rhs_vars=current_vars,
                spline_df=ddf,
                weight_col=weight_col,
                robust=robust,
                time_col=time_col,
                event_col=event_col,
            )
            candidates.append(SelectionResult(current_vars[:], ddf, aic, fml))

        removable = [v for v in current_vars if v not in forced]
        for var in removable:
            test_vars = [v for v in current_vars if v != var]
            if not test_vars:
                continue
            for ddf in df_candidates:
                _, aic, fml = fit_stpm2_and_aic(
                    df_python=df_python,
                    rhs_vars=test_vars,
                    spline_df=ddf,
                    weight_col=weight_col,
                    robust=robust,
                    time_col=time_col,
                    event_col=event_col,
                )
                candidates.append(SelectionResult(test_vars[:], ddf, aic, fml))

        cand_best = None
        for c in candidates:
            if better(c, cand_best):
                cand_best = c

        if cand_best is None:
            break
        if best is not None and cand_best.aic >= best.aic - 1e-12:
            break

        best = cand_best
        current_vars = best.vars_final[:]
        if all(v in forced for v in current_vars):
            break

    if best is None:
        forced_list = [v for v in hard_include if v not in set(hard_exclude)]
        ddf = df_candidates[0]
        _, aic, fml = fit_stpm2_and_aic(
            df_python=df_python,
            rhs_vars=forced_list,
            spline_df=ddf,
            weight_col=weight_col,
            robust=robust,
            time_col=time_col,
            event_col=event_col,
        )
        best = SelectionResult(forced_list, ddf, aic, fml)

    return best

# ============================================================
# Full-strategy bootstrap validation
# ============================================================
def full_strategy_bootstrap_validation_corrected(
    df_full: pd.DataFrame,
    covariates: list[str],
    df_candidates: list[int],
    hard_include: list[str],
    hard_exclude: list[str] | None,
    t0_months: int,
    n_boot: int,
    random_state: int | None = None,
    event_col: str = "Event",
    time_col: str = "Disease_Duration",
    weight_col: str | None = "weight",
    robust: bool = True,
    verbose: bool = True,
):
    hard_exclude = list(hard_exclude or [])
    rng = np.random.default_rng(random_state)
    df_full = df_full.reset_index(drop=True).copy()

    # ---------- Best model on full data ----------
    sel_full = backward_elimination_aic(
        df_python=df_full,
        covariates=covariates,
        df_candidates=df_candidates,
        hard_include=hard_include,
        hard_exclude=hard_exclude,
        weight_col=weight_col,
        robust=robust,
        time_col=time_col,
        event_col=event_col,
    )

    model_best_full, df_full_used, r_full = fit_stpm2_on_dataframe(
        df_python=df_full,
        formula_str=sel_full.formula,
        spline_df=sel_full.spline_df,
        weight_col=weight_col,
        robust=robust,
        time_col=time_col,
        event_col=event_col,
    )

    score_full, info_full = risk_score_stpm2(
        model_best_full,
        r_full,
        vars_final=sel_full.vars_final,
        event_col=event_col,
        weight_col=weight_col,
        sign=None,
        return_details=True
    )

    c_app = cindex_from_risk_score(r_full, score_full, weight_col=weight_col)
    s_app = slope_from_risk_score(r_full, score_full, weight_col=weight_col)
    citl_app = ipcw_citl_from_pred_risk(
        r_full,
        p_pred=1.0 - predict_surv_at_t0(model_best_full, r_full, t0_months),
        t0_months=t0_months,
        event_col=event_col,
        time_col=time_col,
        weight_col=weight_col,
    )

    if verbose:
        print("\n[Apparent] score sign on FULL data:", info_full["chosen_sign"])
        print("[Apparent] mean PI event1/event0:", info_full["mean_pi_event1"], info_full["mean_pi_event0"])
        print("[Apparent] mean score event1/event0:", info_full["mean_score_event1"], info_full["mean_score_event0"])
        print("[Apparent] n used:", len(df_full_used))

    # ---------- Bootstrap ----------
    n = len(df_full)
    c_diffs, s_diffs, citl_diffs = [], [], []
    c_out_list, s_out_list, citl_out_list = [], [], []
    boot_coefs = []

    incl_counts = Counter()
    df_counts = Counter()
    model_counts = Counter()

    n_success = 0
    n_fail = 0

    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        df_boot = df_full.iloc[idx].reset_index(drop=True)

        try:
            sel_boot = backward_elimination_aic(
                df_python=df_boot,
                covariates=covariates,
                df_candidates=df_candidates,
                hard_include=hard_include,
                hard_exclude=hard_exclude,
                weight_col=weight_col,
                robust=robust,
                time_col=time_col,
                event_col=event_col,
            )

            incl_counts.update(sel_boot.vars_final)
            df_counts.update([sel_boot.spline_df])
            model_counts.update([_model_signature(sel_boot.vars_final, sel_boot.spline_df)])

            model_boot, df_boot_used, r_boot = fit_stpm2_on_dataframe(
                df_python=df_boot,
                formula_str=sel_boot.formula,
                spline_df=sel_boot.spline_df,
                weight_col=weight_col,
                robust=robust,
                time_col=time_col,
                event_col=event_col,
            )

            if (
                sel_boot.vars_final == sel_full.vars_final and
                sel_boot.spline_df == sel_full.spline_df
            ):
                bvec = np.asarray(list(ro.r["coef"](model_boot)), dtype=float)
                boot_coefs.append(bvec)

            # In-sample
            score_in, info_in = risk_score_stpm2(
                model_boot,
                r_boot,
                vars_final=sel_boot.vars_final,
                event_col=event_col,
                weight_col=weight_col,
                sign=None,
                return_details=True
            )
            sign_boot = info_in["chosen_sign"]

            c_in = cindex_from_risk_score(r_boot, score_in, weight_col=weight_col)
            s_in = slope_from_risk_score(r_boot, score_in, weight_col=weight_col)
            citl_in = ipcw_citl_from_pred_risk(
                r_boot,
                p_pred=1.0 - predict_surv_at_t0(model_boot, r_boot, t0_months),
                t0_months=t0_months,
                event_col=event_col,
                time_col=time_col,
                weight_col=weight_col,
            )

            # Out-of-bootstrap on cleaned full-data frame used by final model
            score_out = risk_score_stpm2(
                model_boot,
                r_full,
                vars_final=sel_boot.vars_final,
                event_col=event_col,
                weight_col=weight_col,
                sign=sign_boot,
                return_details=False
            )

            c_out = cindex_from_risk_score(r_full, score_out, weight_col=weight_col)
            s_out = slope_from_risk_score(r_full, score_out, weight_col=weight_col)
            citl_out = ipcw_citl_from_pred_risk(
                r_full,
                p_pred=1.0 - predict_surv_at_t0(model_boot, r_full, t0_months),
                t0_months=t0_months,
                event_col=event_col,
                time_col=time_col,
                weight_col=weight_col,
            )

            c_diffs.append(c_in - c_out)
            s_diffs.append(s_in - s_out)
            citl_diffs.append(citl_in - citl_out)

            c_out_list.append(c_out)
            s_out_list.append(s_out)
            citl_out_list.append(citl_out)

            n_success += 1

        except Exception as e:
            n_fail += 1
            if verbose:
                print(f"[Bootstrap {b+1}] failed: {e}")

    # ---------- Optimism correction ----------
    if n_success > 0:
        c_corr = c_app - float(np.nanmean(c_diffs))
        s_corr = s_app - float(np.nanmean(s_diffs))
        citl_corr = citl_app - float(np.nanmean(citl_diffs))

        c_ci = tuple(np.nanpercentile(c_out_list, [2.5, 97.5]).astype(float))
        s_ci = tuple(np.nanpercentile(s_out_list, [2.5, 97.5]).astype(float))
        citl_ci = tuple(np.nanpercentile(citl_out_list, [2.5, 97.5]).astype(float))
    else:
        c_corr, s_corr, citl_corr = c_app, s_app, citl_app
        c_ci = (np.nan, np.nan)
        s_ci = (np.nan, np.nan)
        citl_ci = (np.nan, np.nan)

    denom = max(1, n_success)
    universe = (set(covariates) | set(hard_include)) - set(hard_exclude)
    incl_freq = {v: incl_counts[v] / denom for v in universe}

    res = FullStrategyValidationResult(
        best_vars_full=sel_full.vars_final,
        best_spline_df_full=int(sel_full.spline_df),
        best_aic_full=float(sel_full.aic),
        best_formula_full=sel_full.formula,

        cindex_app=float(c_app),
        slope_app=float(s_app),
        citl_app=float(citl_app),

        cindex_corr=float(c_corr),
        slope_corr=float(s_corr),
        citl_corr=float(citl_corr),

        cindex_ci=(float(c_ci[0]), float(c_ci[1])),
        slope_ci=(float(s_ci[0]), float(s_ci[1])),
        citl_ci=(float(citl_ci[0]), float(citl_ci[1])),

        n_boot_success=int(n_success),
        n_boot_fail=int(n_fail),
    )

    diag = {
        "selection_inclusion_counts": incl_counts,
        "selection_inclusion_freq": incl_freq,
        "selected_df_counts": df_counts,
        "selected_model_counts": model_counts,
        "n_boot_success": n_success,
        "n_boot_fail": n_fail,
        "apparent_score_sign_full": info_full["chosen_sign"],
        "n_full_used": len(df_full_used),
    }

    if verbose:
        print("\nBootstrap reps success/fail:", n_success, n_fail)
        top = sorted(incl_freq.items(), key=lambda kv: kv[1], reverse=True)
        print("\nTop inclusion frequencies:")
        for v, f in top[:10]:
            print(f"  {v}: {f:.2f}")

    return res, diag, model_best_full, boot_coefs

# ============================================================
# External validation
# ============================================================
def external_validate_stpm2_corrected(
    model_final,
    dev_r_df,
    ext_df: pd.DataFrame,
    vars_final: list[str],
    t0_months: int,
    shrinkage: float = 1.0,
    n_bins: int = 5,
    baseline_update: bool = True,
    event_col: str = "Event",
    time_col: str = "Disease_Duration",
    weight_col: str | None = "weight",
    verbose: bool = True,
):
    needed = [time_col, event_col] + list(vars_final)
    if weight_col is not None and weight_col in ext_df.columns:
        needed += [weight_col]

    missing = [c for c in needed if c not in ext_df.columns]
    if missing:
        raise ValueError(f"External df missing columns: {missing}")

    ext_dat = build_analysis_df(
        df_python=ext_df[needed].copy(),
        rhs_vars=vars_final,
        weight_col=weight_col,
        time_col=time_col,
        event_col=event_col,
    )
    r_ext = to_r_df(ext_dat)
    w_ext = _get_weights_from_r(r_ext, weight_col=weight_col)

    # Determine sign on development data
    _, info_dev = risk_score_stpm2(
        model_final, dev_r_df,
        vars_final=vars_final,
        event_col=event_col,
        weight_col=weight_col,
        sign=None,
        return_details=True
    )
    dev_sign = info_dev["chosen_sign"]

    if verbose:
        print("\n[External] Using development-derived score sign:", dev_sign)
        print("[Dev sign check] mean score event1/event0:",
              info_dev["mean_score_event1"], info_dev["mean_score_event0"])

    score_ext = risk_score_stpm2(
        model_final,
        r_ext,
        vars_final=vars_final,
        event_col=event_col,
        weight_col=weight_col,
        sign=dev_sign,
        return_details=False
    )
    score_ext_shrunk = np.asarray(score_ext, float) * float(shrinkage)

    cindex_ext = cindex_from_risk_score(r_ext, score_ext_shrunk, weight_col=weight_col)
    slope_ext  = slope_from_risk_score(r_ext, score_ext_shrunk, weight_col=weight_col)

    S_pred = predict_surv_at_t0(model_final, r_ext, t0_months)
    risk_frozen = 1.0 - S_pred
    S_obs = km_surv_at_t0(r_ext, t0_months, weight_col=weight_col)

    if not np.isfinite(S_obs):
        factor = np.nan
        risk_updated = risk_frozen.copy()
    else:
        S_obs_c = float(np.clip(S_obs, 1e-12, 1 - 1e-12))
        S_bar_c = float(np.clip(weighted_mean_np(S_pred, w_ext), 1e-12, 1 - 1e-12))

        alpha = np.log(-np.log(S_obs_c)) - np.log(-np.log(S_bar_c))
        factor = float(np.exp(alpha))

        if baseline_update:
            S_upd = np.power(S_pred, factor)
            risk_updated = 1.0 - S_upd
        else:
            risk_updated = risk_frozen.copy()

    mean_pred_risk_frozen = weighted_mean_np(risk_frozen, w_ext)
    mean_pred_risk_updated = weighted_mean_np(risk_updated, w_ext)

    citl_frozen_ipcw = ipcw_citl_from_pred_risk(
        r_ext, p_pred=risk_frozen, t0_months=t0_months,
        event_col=event_col, time_col=time_col, weight_col=weight_col
    )
    citl_updated_ipcw = ipcw_citl_from_pred_risk(
        r_ext, p_pred=risk_updated, t0_months=t0_months,
        event_col=event_col, time_col=time_col, weight_col=weight_col
    )

    cal_frozen = calibration_bins_km_robust(
        r_ext, risk_frozen, t0_months, n_bins=n_bins, weight_col=weight_col
    )
    cal_updated = calibration_bins_km_robust(
        r_ext, risk_updated, t0_months, n_bins=n_bins, weight_col=weight_col
    )

    return {
        "t0_months": int(t0_months),
        "vars_final": list(vars_final),
        "shrinkage": float(shrinkage),
        "dev_score_sign": int(dev_sign),

        "cindex_external": float(cindex_ext),
        "slope_external": float(slope_ext),

        "S_obs_km": float(S_obs) if np.isfinite(S_obs) else np.nan,
        "obs_risk_km": float(1.0 - S_obs) if np.isfinite(S_obs) else np.nan,

        "mean_pred_risk_frozen": float(mean_pred_risk_frozen),
        "citl_external_frozen": float(citl_frozen_ipcw),

        "baseline_factor": float(factor) if np.isfinite(factor) else np.nan,
        "mean_pred_risk_updated": float(mean_pred_risk_updated),
        "citl_external_updated": float(citl_updated_ipcw),

        "calibration_table_frozen": cal_frozen,
        "calibration_table_updated": cal_updated,
    }

# ============================================================
# External bootstrap CIs
# ============================================================
def external_bootstrap_cis(
    model_final,
    dev_r_df,
    ext_df: pd.DataFrame,
    vars_final: list[str],
    t0_months: int,
    shrinkage: float = 1.0,
    baseline_update: bool = True,
    n_boot: int = 500,
    random_state: int | None = 123,
    event_col: str = "Event",
    time_col: str = "Disease_Duration",
    weight_col: str | None = "weight",
    return_replicates: bool = False,
):
    def logit(p: float) -> float:
        p = float(np.clip(p, 1e-12, 1 - 1e-12))
        return float(np.log(p/(1-p)))

    def citl(p_obs: float, mean_pred: float) -> float:
        return float(logit(p_obs) - logit(mean_pred))

    _, info_dev = risk_score_stpm2(
        model_final, dev_r_df,
        vars_final=vars_final,
        event_col=event_col,
        weight_col=weight_col,
        sign=None,
        return_details=True
    )
    dev_sign = int(info_dev["chosen_sign"])

    needed = [time_col, event_col] + list(vars_final)
    if weight_col is not None and weight_col in ext_df.columns:
        needed += [weight_col]

    missing = [c for c in needed if c not in ext_df.columns]
    if missing:
        raise ValueError(f"External df missing columns: {missing}")

    rng = np.random.default_rng(random_state)
    n = len(ext_df)

    reps = {
        "cindex": [],
        "slope": [],
        "citl_frozen": [],
        "citl_updated": [],
        "mean_pred_frozen": [],
        "mean_pred_updated": [],
        "obs_risk_km": [],
        "baseline_factor": [],
    }

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_raw = ext_df.iloc[idx].reset_index(drop=True)

        boot = build_analysis_df(
            df_python=boot_raw[needed].copy(),
            rhs_vars=vars_final,
            weight_col=weight_col,
            time_col=time_col,
            event_col=event_col,
        )
        if boot.empty:
            continue

        r_boot = to_r_df(boot)
        w_boot = _get_weights_from_r(r_boot, weight_col=weight_col)

        score = risk_score_stpm2(
            model_final, r_boot,
            vars_final=vars_final,
            event_col=event_col,
            weight_col=weight_col,
            sign=dev_sign,
            return_details=False
        )
        score = np.asarray(score, float) * float(shrinkage)

        reps["cindex"].append(cindex_from_risk_score(r_boot, score, weight_col=weight_col))
        reps["slope"].append(slope_from_risk_score(r_boot, score, weight_col=weight_col))

        S_pred = predict_surv_at_t0(model_final, r_boot, t0_months)
        risk_frozen = 1.0 - S_pred
        mean_pred_frozen = weighted_mean_np(risk_frozen, w_boot)

        S_obs = km_surv_at_t0(r_boot, t0_months, weight_col=weight_col)
        if not np.isfinite(S_obs):
            continue

        p_obs = 1.0 - float(S_obs)
        reps["obs_risk_km"].append(p_obs)
        reps["mean_pred_frozen"].append(mean_pred_frozen)
        reps["citl_frozen"].append(citl(p_obs, mean_pred_frozen))

        S_obs_c = float(np.clip(S_obs, 1e-12, 1 - 1e-12))
        S_bar_c = float(np.clip(weighted_mean_np(S_pred, w_boot), 1e-12, 1 - 1e-12))
        alpha = np.log(-np.log(S_obs_c)) - np.log(-np.log(S_bar_c))
        factor = float(np.exp(alpha))
        reps["baseline_factor"].append(factor)

        if baseline_update:
            S_upd = np.power(S_pred, factor)
            risk_updated = 1.0 - S_upd
        else:
            risk_updated = risk_frozen

        mean_pred_updated = weighted_mean_np(risk_updated, w_boot)
        reps["mean_pred_updated"].append(mean_pred_updated)
        reps["citl_updated"].append(citl(p_obs, mean_pred_updated))

    reps_np = {k: np.asarray(v, float) for k, v in reps.items()}

    def pct_ci(x):
        x = x[np.isfinite(x)]
        if len(x) == 0:
            return (np.nan, np.nan)
        return tuple(np.percentile(x, [2.5, 97.5]).astype(float))

    cis = {k: pct_ci(v) for k, v in reps_np.items()}

    out = {
        "dev_score_sign": dev_sign,
        "n_boot_requested": int(n_boot),
        "n_boot_used_for_km_based": int(len(reps_np["obs_risk_km"])),
        "ci": cis,
        "point_estimate_from_full_external": None,
    }

    if return_replicates:
        out["replicates"] = reps_np

    return out

# ============================================================
# Summary tables
# ============================================================
def make_strategy_summary_table(res_strategy, t0_months: int) -> pd.DataFrame:
    rows = []

    def add(metric, value=None, lo=None, hi=None, digits=3):
        if isinstance(value, (float, int, np.floating)) and value is not None and np.isfinite(value):
            v = f"{float(value):.{digits}f}"
        else:
            v = value
        rows.append({"Metric": metric, "Value": v, "95% CI": ci_str(lo, hi, digits=digits)})

    add("Best model on full data (strategy)", ", ".join(res_strategy.best_vars_full))
    add("Best spline df on full data", res_strategy.best_spline_df_full, digits=0)
    add("AIC (best on full data)", res_strategy.best_aic_full)

    add("C-index (apparent, best model)", res_strategy.cindex_app)
    add("C-index (optimism-corrected, strategy)", res_strategy.cindex_corr,
        res_strategy.cindex_ci[0], res_strategy.cindex_ci[1])

    add("Calibration slope (apparent, best model)", res_strategy.slope_app)
    add("Calibration slope (optimism-corrected, strategy)", res_strategy.slope_corr,
        res_strategy.slope_ci[0], res_strategy.slope_ci[1])

    add(f"CITL at {t0_months} months (apparent, best model)", res_strategy.citl_app)
    add(f"CITL at {t0_months} months (optimism-corrected, strategy)", res_strategy.citl_corr,
        res_strategy.citl_ci[0], res_strategy.citl_ci[1])

    add("Bootstrap reps (success)", res_strategy.n_boot_success, digits=0)
    add("Bootstrap reps (fail)", res_strategy.n_boot_fail, digits=0)

    return pd.DataFrame(rows, columns=["Metric", "Value", "95% CI"])

def make_external_summary_table(
    res_ext: dict,
    t0_months: int,
    boot_cis: dict | None = None,
    digits: int = 3,
) -> pd.DataFrame:
    def fmt_val(x, d=digits):
        if isinstance(x, (float, int, np.floating)) and np.isfinite(x):
            return f"{float(x):.{d}f}"
        return "" if x is None else x

    def fmt_ci(name, d=digits):
        if boot_cis is None:
            return ""
        lo, hi = boot_cis.get("ci", {}).get(name, (np.nan, np.nan))
        if not np.isfinite(lo) or not np.isfinite(hi):
            return ""
        return f"[{lo:.{d}f}, {hi:.{d}f}]"

    rows = []

    def add(metric, value=None, ci_name=None, d=digits):
        rows.append({
            "Metric": metric,
            "Value": fmt_val(value, d),
            "95% CI": fmt_ci(ci_name, d) if ci_name else ""
        })

    add("External time horizon (months)", t0_months, d=0)
    add("Model variables", ", ".join(res_ext.get("vars_final", [])))
    add("Shrinkage factor (uniform)", res_ext.get("shrinkage", np.nan))
    add("C-index (external)", res_ext.get("cindex_external"), "cindex")
    add("Calibration slope (external)", res_ext.get("slope_external"), "slope")
    add(f"Observed KM risk at {t0_months} months", res_ext.get("obs_risk_km"), "obs_risk_km")
    add(f"Mean predicted risk at {t0_months} months (frozen)", res_ext.get("mean_pred_risk_frozen"), "mean_pred_frozen")
    add(f"CITL at {t0_months} months (frozen)", res_ext.get("citl_external_frozen"), "citl_frozen")
    add("Baseline update factor", res_ext.get("baseline_factor"), "baseline_factor")
    add(f"Mean predicted risk at {t0_months} months (baseline-updated)", res_ext.get("mean_pred_risk_updated"), "mean_pred_updated")
    add(f"CITL at {t0_months} months (baseline-updated)", res_ext.get("citl_external_updated"), "citl_updated")

    return pd.DataFrame(rows, columns=["Metric", "Value", "95% CI"])

# ============================================================
# Coefficient tables
# ============================================================
def coef_table_apparent_and_shrunk_2(
    model_best_full,
    vars_final: list[str],
    shrinkage: float,
    digits_beta: int = 3,
    digits_hr: int = 3,
) -> pd.DataFrame:
    def fmt(x, d):
        return "" if (x is None or (isinstance(x, float) and not np.isfinite(x))) else f"{float(x):.{d}f}"

    coef_vec = ro.r["coef"](model_best_full)
    vc = ro.r["vcov"](model_best_full)

    names = list(ro.r["names"](coef_vec))
    b = np.asarray(list(coef_vec), dtype=float)
    idx = {str(n): i for i, n in enumerate(names)}
    vc_np = np.asarray(vc, dtype=float)

    rows = []
    for v in vars_final:
        if v not in idx:
            rows.append({
                "Variable": v,
                "log(HR) apparent": "",
                "95% CI log(HR) apparent": "",
                "HR apparent": "",
                "95% CI HR apparent": "",
                "log(HR) shrunk": "",
                "95% CI log(HR) shrunk": "",
                "HR shrunk": "",
                "95% CI HR shrunk": "",
            })
            continue

        i = idx[v]
        beta = float(b[i])
        var_i = float(vc_np[i, i]) if np.isfinite(vc_np[i, i]) else np.nan
        se = float(np.sqrt(var_i)) if np.isfinite(var_i) and var_i >= 0 else np.nan

        lo = beta - 1.96 * se if np.isfinite(se) else np.nan
        hi = beta + 1.96 * se if np.isfinite(se) else np.nan

        hr = float(np.exp(beta))
        hr_lo = float(np.exp(lo)) if np.isfinite(lo) else np.nan
        hr_hi = float(np.exp(hi)) if np.isfinite(hi) else np.nan

        beta_s = float(beta * shrinkage)
        lo_s = float(lo * shrinkage) if np.isfinite(lo) else np.nan
        hi_s = float(hi * shrinkage) if np.isfinite(hi) else np.nan

        hr_s = float(np.exp(beta_s))
        hr_s_lo = float(np.exp(lo_s)) if np.isfinite(lo_s) else np.nan
        hr_s_hi = float(np.exp(hi_s)) if np.isfinite(hi_s) else np.nan

        rows.append({
            "Variable": v,
            "log(HR) apparent": fmt(beta, digits_beta),
            "95% CI log(HR) apparent": ci_str(lo, hi, digits=digits_beta),
            "HR apparent": fmt(hr, digits_hr),
            "95% CI HR apparent": ci_str(hr_lo, hr_hi, digits=digits_hr),
            "log(HR) shrunk": fmt(beta_s, digits_beta),
            "95% CI log(HR) shrunk": ci_str(lo_s, hi_s, digits=digits_beta),
            "HR shrunk": fmt(hr_s, digits_hr),
            "95% CI HR shrunk": ci_str(hr_s_lo, hr_s_hi, digits=digits_hr),
        })

    return pd.DataFrame(rows)

# ============================================================
# Calibration plotting / tidying
# ============================================================
def tidy_calibration_table(cal_df: pd.DataFrame) -> pd.DataFrame:
    out = cal_df.copy()
    out["Observed 95% CI"] = out.apply(lambda r: ci_str(float(r["obs_lo"]), float(r["obs_hi"]), digits=3), axis=1)
    out["mean_pred"] = out["mean_pred"].map(lambda v: f"{float(v):.3f}")
    out["obs_km"] = out["obs_km"].map(lambda v: f"{float(v):.3f}")
    out = out[["bin", "n", "mean_pred", "obs_km", "Observed 95% CI"]]
    out.columns = ["Bin", "N", "Mean predicted risk", "Observed risk (KM)", "Observed 95% CI"]
    return out

def tidy_calibration_df(cal_df: pd.DataFrame) -> pd.DataFrame:
    df = cal_df.copy()
    for c in ["mean_pred", "obs_km", "obs_lo", "obs_hi"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["mean_pred", "obs_km", "obs_lo", "obs_hi"]).reset_index(drop=True)
    return df

def plot_calibration_table(
    cal_df: pd.DataFrame,
    title: str,
    percent: bool = True,
    show_45deg: bool = True,
    ylim: tuple[float, float] | None = None,
):
    if cal_df is None or len(cal_df) == 0:
        raise ValueError("Calibration table is empty.")

    x = cal_df["mean_pred"].to_numpy(dtype=float)
    y = cal_df["obs_km"].to_numpy(dtype=float)
    lo = cal_df["obs_lo"].to_numpy(dtype=float)
    hi = cal_df["obs_hi"].to_numpy(dtype=float)

    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(lo) & np.isfinite(hi)
    x, y, lo, hi = x[ok], y[ok], lo[ok], hi[ok]

    yerr_low = np.maximum(y - lo, 0.0)
    yerr_high = np.maximum(hi - y, 0.0)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.errorbar(x, y, yerr=np.vstack([yerr_low, yerr_high]), fmt="o", capsize=3)

    maxv = float(np.nanmax(np.r_[x, y, hi])) if len(x) else 1.0
    maxv = max(maxv, 1e-6)

    if show_45deg:
        ax.plot([0, maxv], [0, maxv], "--")

    ax.set_xlim(0, maxv)
    ax.set_ylim(0, maxv if ylim is None else ylim[1])
    if ylim is not None:
        ax.set_ylim(ylim[0], ylim[1])

    ax.set_xlabel("Predicted risk")
    ax.set_ylabel("Observed risk")
    ax.set_title(title)

    if percent:
        fmt = FuncFormatter(lambda v, p: f"{100*v:.0f}%")
        ax.xaxis.set_major_formatter(fmt)
        ax.yaxis.set_major_formatter(fmt)

    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig, ax

# ============================================================
# QUICK USAGE
# ============================================================
# covariates = ["Age", "Sex_Male", "Diagnostic_Delay", "Onset_Limb", "Vital_capacity"]
# forced_vars = ["Age", "Sex_Male", "Diagnostic_Delay", "Onset_Limb"]
# df_candidates = [1, 2, 3]
#
# dev = prep_df(dat_set, covariates, weight_col="weight")
#
# res_strategy, diag, model_best_full, boot_coefs = full_strategy_bootstrap_validation_corrected(
#     df_full=dev,
#     covariates=covariates,
#     df_candidates=df_candidates,
#     hard_include=forced_vars,
#     hard_exclude=None,
#     t0_months=24,
#     n_boot=50,
#     random_state=4,
#     weight_col="weight",
#     robust=True,
#     verbose=True,
# )
#
# # Important: apparent evaluation should use the SAME cleaned dataset
# model_eval, dev_used, r_dev_used = fit_stpm2_on_dataframe(
#     df_python=dev,
#     formula_str=res_strategy.best_formula_full,
#     spline_df=res_strategy.best_spline_df_full,
#     weight_col="weight",
#     robust=True,
# )
#
# apparent = evaluate_stpm2_apparent(
#     model_best_full,
#     r_dev_used,
#     vars_final=res_strategy.best_vars_full,
#     t0_months=24,
#     weight_col="weight",
#     verbose=True
# )

#### **Plotting functions**

In [ ]:
_r_predict_surv_curve = R("""
function(model, newdata, times, coef_override=NULL, timevar="Disease_Duration"){

  m2 <- model

  # stpm2 from rstpm2 is an S4 object; update slots safely
  if(!is.null(coef_override)){
    co <- as.numeric(coef_override)

    # Ensure names exist (helps matching)
    # (coef(m2) and m2@fullcoef usually have names already)
    if(length(co) == length(m2@fullcoef)){
      m2@fullcoef <- co

      # If coef slot exists and matches some part of fullcoef, update it too
      if(length(m2@coef) > 0 && !is.null(names(m2@coef)) && !is.null(names(m2@fullcoef))){
        pos <- match(names(m2@coef), names(m2@fullcoef))
        ok <- which(!is.na(pos))
        if(length(ok) > 0){
          m2@coef <- m2@fullcoef[pos]
        }
      }

    } else if(length(co) == length(m2@coef)){
      # update coef slot
      m2@coef <- co

      # also update matching entries in fullcoef by name
      if(!is.null(names(m2@coef)) && !is.null(names(m2@fullcoef))){
        pos <- match(names(m2@coef), names(m2@fullcoef))
        ok <- which(!is.na(pos))
        if(length(ok) > 0){
          fc <- m2@fullcoef
          fc[pos[ok]] <- co[ok]
          m2@fullcoef <- fc
        }
      }
    } else {
      stop("coef_override length does not match length(m2@coef) nor length(m2@fullcoef)")
    }
  }

  # Predict S(t) for each time; return n x T
  out <- sapply(times, function(tt){
    tmp <- newdata
    tmp[[timevar]] <- rep(as.numeric(tt), nrow(tmp))
    as.numeric(predict(m2, newdata=tmp, type="surv", timevar=timevar))
  })

  t(out)
}
""")

def km_curve(df: pd.DataFrame, label: str, weight_col: str | None = None):
    km = KaplanMeierFitter()

    if weight_col is not None and weight_col in df.columns:
        km.fit(
            durations=df["Disease_Duration"],
            event_observed=(df["Event"] == 1),
            weights=df[weight_col].astype(float),
            label=label
        )
    else:
        km.fit(
            durations=df["Disease_Duration"],
            event_observed=(df["Event"] == 1),
            label=label
        )
    return km


def plot_km_treated_vs_hist_placebo(treated_df, hist_placebo_df, x_max=24, weight_col: str | None = None):
    km_t = km_curve(treated_df, "Observed treated (KM)", weight_col=weight_col)
    km_p = km_curve(hist_placebo_df, "Historical placebo (KM)", weight_col=weight_col)

    ax = km_t.plot_survival_function(ci_show=True)
    km_p.plot_survival_function(ci_show=True, ax=ax)

    plt.xlim(0, x_max)
    plt.xlabel("Time (months)")
    plt.ylabel("Survival probability")
    plt.title("Observed treated vs historical placebo")
    plt.show()

def virtual_placebo_survival_curve(
    model_placebo,
    treated_df: pd.DataFrame,
    vars_final: list[str],
    time_grid: np.ndarray,
    shrinkage: float = 1.0,
    coef_boot: np.ndarray | None = None,
    weight_col: str | None = None,
):
    needed = ["Disease_Duration"] + list(vars_final)
    if weight_col is not None and weight_col in treated_df.columns:
        needed = needed + [weight_col]

    tmp = treated_df.copy()
    for c in ["Disease_Duration"] + list(vars_final):
        if c not in tmp.columns:
            raise ValueError(f"treated_df missing column: {c}")

    r_treated = to_r_df(tmp[needed])

    b = np.asarray(list(ro.r["coef"](model_placebo)), dtype=float)
    names = list(map(str, ro.r["names"](ro.r["coef"](model_placebo))))
    idx = {n: i for i, n in enumerate(names)}

    if coef_boot is not None:
        b_use = np.asarray(coef_boot, dtype=float).copy()
        if len(b_use) != len(b):
            raise ValueError("coef_boot must match length of coef(model_placebo)")
    else:
        b_use = b.copy()

    for v in vars_final:
        if v in idx:
            b_use[idx[v]] = b_use[idx[v]] * float(shrinkage)

    S_mat = np.asarray(_r_predict_surv_curve(
        model_placebo,
        r_treated,
        ro.FloatVector(np.asarray(time_grid, float).tolist()),
        ro.FloatVector(b_use.tolist())
    ), dtype=float)

    T = len(time_grid)
    n = len(treated_df)

    if S_mat.shape == (T, n):
        S_mat = S_mat.T
    elif S_mat.shape != (n, T):
        raise ValueError(f"Unexpected S_mat shape {S_mat.shape}, expected {(n,T)} or {(T,n)}")

    if weight_col is not None and weight_col in treated_df.columns:
        w = treated_df[weight_col].to_numpy(float)
        avg_S = np.average(S_mat, axis=0, weights=w)
    else:
        avg_S = np.mean(S_mat, axis=0)

    return avg_S, S_mat


def plot_virtual_placebo_vs_treated(
    model_placebo,
    treated_df,
    vars_final,
    time_grid,
    shrinkage=1.0,
    x_max=24,
    weight_col: str | None = None,
):
    avg_S, _ = virtual_placebo_survival_curve(
        model_placebo=model_placebo,
        treated_df=treated_df,
        vars_final=vars_final,
        time_grid=time_grid,
        shrinkage=shrinkage,
        coef_boot=None,
        weight_col=weight_col,
    )

    km_t = km_curve(treated_df, "Observed treated (KM)", weight_col=weight_col)

    plt.figure()
    plt.plot(time_grid, avg_S, label="Virtual placebo (avg predicted)")
    ax = km_t.plot_survival_function(ci_show=True)
    plt.xlim(-0.01, x_max + 5)
    plt.xlabel("Time (months)")
    plt.ylabel("Survival probability")
    plt.title("Virtual placebo (treated) vs observed treated")
    plt.legend()
    plt.ylim(-0.01, 1.1)
    plt.grid(True)
    plt.show()


#### Apply the functions

In [ ]:
comb_weight.head(2)

In [ ]:
dat_set = comb_weight[comb_weight['Study_Arm_Placebo'] == 1]

# covariates = ["Onset_Limb", "Sex_Male", "Diagnostic_Delay",# "Vital_capacity",
#                 'Sex_onset',   "Age_onset",  "Age_Sex", "Age_sq"]#, "Sex_VC", "Age_VC","Onset_VC"]

# def drop_constant_columns(df, cols):
#     return [c for c in cols if df[c].nunique() > 1]

# current_vars = drop_constant_columns(dat_set, covariates)
# print(current_vars)


In [ ]:
# covariates = current_vars # ["Age", "Sex_Male", "Diagnostic_Delay", "Onset_Limb", "Vital_capacity"]
covariates =  ["Age", "Sex_Male", "Diagnostic_Delay", "Onset_Limb"] #, "Vital_capacity"]

forced_vars = ["Age", "Sex_Male",  "Onset_Limb", "Diagnostic_Delay"]#, "Vital_capacity"]

df_candidates = [1,2,3]   # recommended now that df=3 showed up
t0_months = 25
n_boot_full_strategy = 10
seed_rand = 4

# ============================================================
# USAGE 
# ============================================================
dev = prep_df(dat_set, covariates, weight_col='weight')

res_strategy, diag, model_best_full, boot_coefs = full_strategy_bootstrap_validation_corrected(
    df_full=dev,
    covariates=covariates,
    df_candidates=df_candidates,
    hard_include=forced_vars,
    hard_exclude=None,
    t0_months=24,
    n_boot=5,
    random_state=4,
    weight_col="weight",
    robust=True,
    verbose=True,
)
#
# 
# ============================================================
print("Best model on full data:", res_strategy.best_vars_full)


In [ ]:
comb_weight.isna().sum()

In [ ]:
# dat_set = comb_weight[comb_weight['Study_Arm_Placebo'] == 1]
ext_df = comb_weight[comb_weight['Study_Arm_Placebo'] == 0]
ext_df.head(2)
# dev = prep_df(dat_set, covariates, weight_col="weight")
ext = prep_df(ext_df, covariates, weight_col="weight")

In [ ]:
b_bins = 5

# Development R df with the right columns
vars_final = res_strategy.best_vars_full
r_dev = to_r_df(dev[["Disease_Duration", "Event"] + vars_final])

# Refit final model on development data using the selected formula + df
model_final = rstpm2.stpm2( Formula(res_strategy.best_formula_full),
    data=r_dev, df=res_strategy.best_spline_df_full)

# Shrinkage (will be <=1; often 1.0 if slope_corr < 1)
shrinkage = uniform_shrinkage_from_slope(res_strategy.slope_corr)

res_ext = external_validate_stpm2_corrected(
    model_final=model_final,
    dev_r_df=r_dev,
    ext_df=ext,
    vars_final=vars_final,
    t0_months=t0_months,
    shrinkage=shrinkage,
    n_bins=b_bins,
    baseline_update=True,
    weight_col="weight",
    verbose=True
)

boot_cis = external_bootstrap_cis(
    model_final=model_final,
    dev_r_df=r_dev,
    ext_df=ext,
    vars_final=vars_final,
    t0_months=t0_months,
    shrinkage=shrinkage,
    baseline_update=True,
    n_boot=n_boot_full_strategy,
    random_state=seed_rand,
    weight_col="weight",
)

In [ ]:
# 1) INTERNAL strategy summary (CI as separate column)
print(f"\nINTERNAL VALIDATION SUMMARY (STRATEGY-LEVEL) ")
internal_summary_tbl = make_strategy_summary_table(res_strategy, t0_months=t0_months)
display(internal_summary_tbl)
# internal_summary_tbl.to_excel("/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/Results/lica_placebo_INTERNAL_VALIDATION_SUMMARY.xlsx")

# 2) Coefficients summary (apparent vs updated/shrunk)
print("\nCOEFFICIENT SUMMARY (apparent vs updated/shrunk)")
vars_final = res_strategy.best_vars_full
# print()
coef_summary_tbl_2 = coef_table_apparent_and_shrunk_2(model_best_full, vars_final=vars_final, shrinkage=shrinkage)
display(coef_summary_tbl_2)
# coef_summary_tbl_2.to_excel("/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/Results/lica_placebo_COEFFICIENT_SUMMARY.xlsx")

# 3) EXTERNAL summary (CI column included where applicable)
print(f"\nEXTERNAL VALIDATION SUMMARY ")
ext_summary_tbl = make_external_summary_table(res_ext=res_ext, t0_months=t0_months, boot_cis=boot_cis)
display(ext_summary_tbl)
# ext_summary_tbl.to_excel("/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/Results/lica_placebo_nEXTERNAL_VALIDATION_SUMMARY.xlsx")


In [ ]:
hist_placebo_df = prep_df(dat_set.copy(), covariates, weight_col='weight')
obs_treated_df  = prep_df(ext_df.copy(), covariates, weight_col='weight')
obs_placebo_df  = prep_df(dat_set.copy(), covariates, weight_col='weight')

time_max = hist_placebo_df["Disease_Duration"].max()
time_max = hist_placebo_df["Disease_Duration"].max()
time_grid = np.linspace(0.05, time_max, 100)  # avoid 0 exactly sometimes

print(obs_treated_df.columns)
print(obs_placebo_df.columns)

In [ ]:
def plot_hybrid(
    model_placebo,
    obs_treated_df,
    obs_placebo_df,
    vars_final,
    time_grid,
    shrinkage=1.0,
    x_max=24,
    model_data='model data',
    applied_data='applied data',
    obs_treated='trt_name',
    obs_placebo='pla_name',
    weight_col: str | None = None,
):
    avg_S, _ = virtual_placebo_survival_curve(
        model_placebo=model_placebo,
        treated_df=obs_treated_df,
        vars_final=vars_final,
        time_grid=time_grid,
        shrinkage=shrinkage,
        coef_boot=None,
        weight_col = weight_col,
    )

    km_t = km_curve(obs_treated_df, f"Observed treated {obs_treated}", weight_col=weight_col)
    km_p = km_curve(obs_placebo_df, f"Observed placebo {obs_placebo}", weight_col=weight_col)

    ax = km_p.plot_survival_function(ci_show=False)
    km_t.plot_survival_function(ci_show=False, ax=ax)
    plt.plot(time_grid, avg_S, label=f"Virtual placebo {applied_data}")

    plt.xlim(0, x_max)
    plt.ylim(-0.05, 1.05)
    plt.xlabel("Disease duration (months)")
    plt.ylabel("Survival probability")
    plt.title(f"Placebo model {model_data}")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
plot_hybrid(
    model_placebo=model_final,
    obs_treated_df=obs_treated_df,
    obs_placebo_df=obs_placebo_df,
    vars_final=vars_final,
    time_grid=time_grid,
    shrinkage=shrinkage,
    x_max=obs_treated_df["Disease_Duration"].max() + 2,
    model_data='(PROACT LICALS Unweighted)',
    applied_data='(LICALS)',
    obs_treated='(LICALS)',
    obs_placebo='(PROACT LICALS)',
    weight_col=None,
)

In [ ]:
plot_hybrid(
    model_placebo=model_final,
    obs_treated_df=obs_treated_df,
    obs_placebo_df=obs_placebo_df,
    vars_final=vars_final,
    time_grid=time_grid,
    shrinkage=shrinkage,
    x_max=obs_treated_df["Disease_Duration"].max() + 2,
    model_data='(PROACT LICALS Weighted)',
    applied_data='(LICALS)',
    obs_treated='(LICALS)',
    obs_placebo='(PROACT LICALS)',
    weight_col="weight"
)

In [ ]:
print(obs_treated_df.columns)
print(obs_placebo_df.columns)

In [ ]:
avg_S_unw, S_mat = virtual_placebo_survival_curve(
    model_placebo=model_final,
    treated_df=obs_treated_df,
    vars_final=vars_final,
    time_grid=time_grid,
    shrinkage=shrinkage,
    coef_boot=None,
    weight_col=None
)

avg_S_w, _ = virtual_placebo_survival_curve(
    model_placebo=model_final,
    treated_df=obs_treated_df,
    vars_final=vars_final,
    time_grid=time_grid,
    shrinkage=shrinkage,
    coef_boot=None,
    weight_col="weight"
)

print("max abs difference:", np.max(np.abs(avg_S_unw - avg_S_w)))

plt.figure(figsize=(6,4))
plt.plot(time_grid, avg_S_unw, label="Virtual placebo unweighted")
plt.plot(time_grid, avg_S_w, label="Virtual placebo weighted")
plt.legend()
plt.grid(True)
plt.show()